In [0]:
# TITLE: Postal Demo Gold Layer Build
#
# PURPOSE
# - Build a rerunnable Gold layer from workspace.silver into workspace.gold.
# - Materialize a Power BI-friendly denormalized star schema as physical Delta tables.
# - Validate grain, key uniqueness, and orphan foreign keys after the build.
#
# DESIGN REVIEW
# Gold dimensions
# - dim_date: one row per calendar date
# - dim_customer: one row per customer.customer_id
# - dim_business: one row per business.business_id
# - dim_package: one row per package.package_id
# - dim_facility: one row per facility.facility_id
# - dim_department: one row per departments.department_id
# - dim_employee: one row per employee.employee_id
# - dim_geography: one row per territory.territory_id
# - dim_service_type: one row per service_type.service_type_id
# - dim_package_status: one row per package_status.package_status_id
# - dim_package_flow_type: one row per package_flow_type.package_flow_type_id
# - dim_incident_type: one row per incident_type.incident_type_id
# - dim_refund_status: one row per distinct refunds.refund_status
# - dim_smartlocker: one row per smartlocker.locker_id
#
# Gold facts
# - fact_package: one row per package.package_id
# - fact_package_movement: one row per package_movement.package_movement_id
# - fact_shipping_revenue_cost: one row per shipping_cost record
# - fact_incident: one row per incident.incident_id
# - fact_refund: one row per refunds.refund_id
# - fact_smartlocker_assignment: one row per package_to_locker.package_id
#
# Required Silver sources by major area
# - Shared lookups: service_type, package_status, package_flow_type, incident_type
# - Geography/facility/employee: territory, zip_geo (optional enrichment), facility, facility_type,
#   departments, department_type, employee, lockerlocations (optional enrichment), smartlocker
# - Customer/business/package: customer, business, package, shippingdetails, package_route_plan (optional enrichment)
# - Facts: package_movement, package_movement_event_type, shipping_cost, incident, incident_status,
#   incident_severity, refunds, package_to_locker, lockerassignment
#
# Fact join paths
# - fact_package: package -> dim_package, shippingdetails, shipping_cost, package_route_plan,
#   movement aggregates, incident aggregates, refund aggregates
# - fact_package_movement: package_movement -> dim_package, dim_facility (event/from/to),
#   dim_employee, package_movement_event_type, package, package_status
# - fact_shipping_revenue_cost: shipping_cost -> dim_package, package, shippingdetails,
#   package_route_plan, dim_customer/dim_business, dim_facility, dim_geography
# - fact_incident: incident -> dim_package, dim_incident_type, dim_employee, dim_facility,
#   incident_status, incident_severity
# - fact_refund: refunds -> incident -> dim_package, dim_refund_status, dim_incident_type,
#   dim_employee, dim_facility, dim_customer
# - fact_smartlocker_assignment: package_to_locker -> lockerassignment -> smartlocker ->
#   lockerlocations -> dim_facility/dim_geography, plus dim_package/dim_customer
#
# Fact row-count validation rules
# - fact_package row count must equal package row count
# - fact_package_movement row count must equal package_movement row count
# - fact_shipping_revenue_cost row count must equal shipping_cost row count
# - fact_incident row count must equal incident row count
# - fact_refund row count must equal refunds row count
# - fact_smartlocker_assignment row count must equal package_to_locker row count
#
# Verified ambiguities / TODOs discovered from source files
# - source_to_gold_mapping_template.csv is only a starter template, not a final mapping contract
# - shipping_cost has no shipping_cost_id, so the fact uses a deterministic hash key
# - refund status has no lookup table, so dim_refund_status is derived from distinct refunds.refund_status values
# - package does not store origin/destination facility IDs, so route facility enrichment uses package_route_plan when available
# - lockerassignment has no package_id, so fact_smartlocker_assignment uses package_to_locker as the base grain
# - zip_geo exists in the sample files and Bronze-to-Silver notebook, but may be absent from some metadata exports;
#   geography enrichment will warn and degrade gracefully if zip_geo is unavailable at runtime

In [0]:
catalog_name = "postal_databricks"

silver_schema = "silver"
gold_schema = "gold"

# Gold schema ADLS managed location.
# This assumes you created an external location covering the gold container.
gold_managed_location = "abfss://gold@postalsg.dfs.core.windows.net/postal_bi_system/"

#For descriptive ADLS file path names
gold_base_path = "abfss://gold@postalsg.dfs.core.windows.net/gold_tables"

write_mode = "overwrite"
enable_overwrite = True

# Keep these disabled for now during development.
enable_optimize = False
enable_vacuum = False

# Validation behavior
strict_required_key_checks = False
display_row_limit = 100

In [0]:
# Imports and helper functions

from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T


validation_results = []
orphan_results = []
table_row_counts = []
warnings_log = []
gold_tables_created = []


def fqtn(table_name: str, schema_name: str = gold_schema, catalog: str = catalog_name) -> str:
    return f"`{catalog}`.`{schema_name}`.`{table_name}`"


def silver_fqtn(table_name: str) -> str:
    return fqtn(table_name, schema_name=silver_schema)


def table_exists(table_name: str, schema_name: str = silver_schema, catalog: str = catalog_name) -> bool:
    return spark.catalog.tableExists(f"{catalog}.{schema_name}.{table_name}")


def record_warning(message: str) -> None:
    warnings_log.append({"warning_message": message})
    print(f"WARNING: {message}")


def record_validation(check_name: str, table_name: str, status: str, metric_value, details: str) -> None:
    validation_results.append(
        {
            "check_name": check_name,
            "table_name": table_name,
            "status": status,
            "metric_value": metric_value,
            "details": details,
        }
    )


def record_orphan_check(fact_table: str, fact_column: str, dim_table: str, orphan_count: int, details: str) -> None:
    orphan_results.append(
        {
            "fact_table": fact_table,
            "fact_column": fact_column,
            "dimension_table": dim_table,
            "orphan_count": orphan_count,
            "details": details,
        }
    )


def read_silver_table(table_name: str, required: bool = True) -> DataFrame | None:
    if not table_exists(table_name, schema_name=silver_schema, catalog=catalog_name):
        if required:
            raise ValueError(f"Required Silver table is missing: {catalog_name}.{silver_schema}.{table_name}")
        record_warning(f"Optional Silver table is missing and will be skipped: {catalog_name}.{silver_schema}.{table_name}")
        return None
    return spark.table(silver_fqtn(table_name))


def assert_required_columns(df: DataFrame | None, table_name: str, required_columns: list[str]) -> None:
    if df is None:
        return
    missing_columns = [column_name for column_name in required_columns if column_name not in df.columns]
    if missing_columns:
        raise ValueError(
            f"Required columns missing from {catalog_name}.{silver_schema}.{table_name}: {', '.join(missing_columns)}"
        )


def normalized_str(column_expr) -> F.Column:
    return F.coalesce(column_expr.cast("string"), F.lit("__null__"))


def normalized_binary(column_expr) -> F.Column:
    return F.coalesce(F.lower(F.hex(column_expr)), F.lit("__null__"))


def deterministic_key(*parts) -> F.Column:
    return F.xxhash64(*parts).cast("long")


def with_gold_audit_columns(df: DataFrame) -> DataFrame:
    current_ts = F.current_timestamp()
    return df.withColumn("gold_created_at", current_ts).withColumn("gold_updated_at", current_ts)


def write_gold_table(df: DataFrame, table_name: str) -> DataFrame:
    table_path = f"{gold_base_path.rstrip('/')}/{table_name}"

    (
        df.write.format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .option("path", table_path)
        .saveAsTable(fqtn(table_name))
    )

    gold_tables_created.append(table_name)

    return spark.table(fqtn(table_name))


def row_count(df: DataFrame) -> int:
    return df.count()


def validate_row_count(table_name: str, actual_count: int, expected_count: int, check_name: str = "row_count_match") -> None:
    status = "PASS" if actual_count == expected_count else "FAIL"
    record_validation(
        check_name=check_name,
        table_name=table_name,
        status=status,
        metric_value=actual_count,
        details=f"expected_row_count={expected_count}",
    )


def validate_non_null(df: DataFrame, table_name: str, column_name: str) -> None:
    null_count = df.filter(F.col(column_name).isNull()).count()
    status = "PASS" if null_count == 0 else "FAIL"
    record_validation(
        check_name="non_null_key_check",
        table_name=table_name,
        status=status,
        metric_value=null_count,
        details=f"column={column_name}",
    )


def validate_unique_keys(df: DataFrame, table_name: str, key_columns: list[str], check_name: str = "duplicate_key_check") -> None:
    duplicate_groups = (
        df.groupBy(*[F.col(column_name) for column_name in key_columns])
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    status = "PASS" if duplicate_groups == 0 else "FAIL"
    record_validation(
        check_name=check_name,
        table_name=table_name,
        status=status,
        metric_value=duplicate_groups,
        details=f"columns={','.join(key_columns)}",
    )


def validate_required_foreign_key(df: DataFrame, table_name: str, column_name: str) -> None:
    unresolved_count = df.filter(F.col(column_name).isNull()).count()
    status = "PASS" if unresolved_count == 0 else ("WARN" if not strict_required_key_checks else "FAIL")
    record_validation(
        check_name="required_foreign_key_resolution",
        table_name=table_name,
        status=status,
        metric_value=unresolved_count,
        details=f"column={column_name}",
    )


def validate_orphan_keys(
    fact_df: DataFrame,
    fact_table: str,
    fact_column: str,
    dim_df: DataFrame,
    dim_column: str,
    dim_table_name: str,
    required_only: bool = False,
) -> None:
    candidate_df = fact_df.filter(F.col(fact_column).isNotNull()) if required_only else fact_df
    orphan_count = (
        candidate_df.alias("fact")
        .join(
            dim_df.select(F.col(dim_column).alias("_dim_key")).alias("dim"),
            F.col(f"fact.{fact_column}") == F.col("dim._dim_key"),
            "left",
        )
        .filter(F.col(f"fact.{fact_column}").isNotNull() & F.col("dim._dim_key").isNull())
        .count()
    )
    record_orphan_check(
        fact_table=fact_table,
        fact_column=fact_column,
        dim_table=dim_table_name,
        orphan_count=orphan_count,
        details=f"dimension_key_column={dim_column}",
    )


def date_key_from_timestamp(column_name: str) -> F.Column:
    return F.when(F.col(column_name).isNull(), F.lit(None).cast("int")).otherwise(
        F.date_format(F.to_date(F.col(column_name)), "yyyyMMdd").cast("int")
    )


def add_table_row_count(table_name: str, df: DataFrame) -> None:
    table_row_counts.append({"table_name": table_name, "row_count": df.count()})


def safe_bool_from_name(name_col: str, expected_values: list[str]) -> F.Column:
    lowered = F.lower(F.coalesce(F.col(name_col), F.lit("")))
    return lowered.isin([value.lower() for value in expected_values])

def bool_from_tinyint(column_expr) -> F.Column:
    return (
        F.when(column_expr.isNull(), F.lit(False))
        .when(column_expr.cast("int") == 1, F.lit(True))
        .otherwise(F.lit(False))
    )

In [0]:
# Create Gold schema if needed.
# This makes new managed Gold Delta tables physically land in the ADLS gold container.

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{gold_schema}`
MANAGED LOCATION '{gold_managed_location}'
""")

DataFrame[]

In [0]:
# Load Silver tables

customer_raw = read_silver_table("customer")
assert_required_columns(
    customer_raw,
    "customer",
    [
        "customer_id",
        "first_name",
        "middle_initial",
        "last_name",
        "street_address",
        "county",
        "city",
        "state_code",
        "zip_code",
        "territory_id",
        "phone_number",
        "email",
        "created_at",
        "updated_at",
        "user_id",
        "preferred_facility_id",
        "birth_date",
        "marital_status",
        "gender",
        "email_address",
        "annual_income",
        "total_children",
        "education_level",
        "occupation",
        "home_owner",
    ],
)
customer = customer_raw.select(
    "customer_id",
    "first_name",
    "middle_initial",
    "last_name",
    "street_address",
    "county",
    "city",
    "state_code",
    "zip_code",
    "territory_id",
    "phone_number",
    "email",
    "created_at",
    "updated_at",
    "user_id",
    "preferred_facility_id",
    "birth_date",
    "marital_status",
    "gender",
    "email_address",
    "annual_income",
    "total_children",
    "education_level",
    "occupation",
    "home_owner",
)

business_raw = read_silver_table("business")
assert_required_columns(
    business_raw,
    "business",
    [
        "business_id",
        "business_name",
        "street_address",
        "county",
        "city",
        "state_code",
        "zip_code",
        "territory_id",
        "phone_number",
        "email",
        "created_at",
        "updated_at",
        "preferred_facility_id",
    ],
)
business = business_raw.select(
    "business_id",
    "business_name",
    "street_address",
    "county",
    "city",
    "state_code",
    "zip_code",
    "territory_id",
    "phone_number",
    "email",
    "created_at",
    "updated_at",
    "preferred_facility_id",
)

facility_raw = read_silver_table("facility")
assert_required_columns(
    facility_raw,
    "facility",
    [
        "facility_id",
        "facility_type_id",
        "manager_employee_id",
        "facility_name",
        "street_address",
        "county",
        "city",
        "state_code",
        "zip_code",
        "facility_department_prefix",
        "territory_id",
        "created_at",
        "updated_at",
    ],
)
facility = facility_raw.select(
    "facility_id",
    "facility_type_id",
    "manager_employee_id",
    "facility_name",
    "street_address",
    "county",
    "city",
    "state_code",
    "zip_code",
    "facility_department_prefix",
    "territory_id",
    "created_at",
    "updated_at",
)

facility_type_raw = read_silver_table("facility_type")
assert_required_columns(
    facility_type_raw,
    "facility_type",
    [
        "facility_type_id",
        "facility_type_code",
        "facility_type_name",
        "description",
        "is_customer_facing",
        "handles_retail",
        "handles_processing",
        "handles_distribution",
        "handles_local_delivery",
        "is_active",
        "created_at",
        "updated_at",
    ],
)
facility_type = facility_type_raw.select(
    "facility_type_id",
    "facility_type_code",
    "facility_type_name",
    "description",
    "is_customer_facing",
    "handles_retail",
    "handles_processing",
    "handles_distribution",
    "handles_local_delivery",
    "is_active",
    "created_at",
    "updated_at",
)

departments_raw = read_silver_table("departments")
assert_required_columns(
    departments_raw,
    "departments",
    [
        "department_id",
        "department_name",
        "department_type_id",
        "manager_employee_id",
        "manager_start_date",
        "facility_id",
        "created_at",
        "updated_at",
    ],
)
departments = departments_raw.select(
    "department_id",
    "department_name",
    "department_type_id",
    "manager_employee_id",
    "manager_start_date",
    "facility_id",
    "created_at",
    "updated_at",
)

department_type_raw = read_silver_table("department_type")
assert_required_columns(
    department_type_raw,
    "department_type",
    [
        "department_type_id",
        "department_type_code",
        "department_type_name",
        "description",
        "is_active",
        "created_at",
        "updated_at",
    ],
)
department_type = department_type_raw.select(
    "department_type_id",
    "department_type_code",
    "department_type_name",
    "description",
    "is_active",
    "created_at",
    "updated_at",
)

employee_raw = read_silver_table("employee")
assert_required_columns(
    employee_raw,
    "employee",
    [
        "employee_id",
        "department_id",
        "full_name",
        "phone_number",
        "email",
        "street_address",
        "job_title",
        "salary",
        "hours_worked",
        "manager_employee_id",
        "created_at",
        "updated_at",
        "user_id",
    ],
)
employee = employee_raw.select(
    "employee_id",
    "department_id",
    "full_name",
    "phone_number",
    "email",
    "street_address",
    "job_title",
    "salary",
    "hours_worked",
    "manager_employee_id",
    "created_at",
    "updated_at",
    "user_id",
)

territory_raw = read_silver_table("territory")
assert_required_columns(
    territory_raw,
    "territory",
    ["territory_id", "state", "city", "county", "zip_code", "created_at", "updated_at"],
)
territory = territory_raw.select(
    "territory_id",
    "state",
    "city",
    "county",
    "zip_code",
    "created_at",
    "updated_at",
)

zip_geo_raw = read_silver_table("zip_geo", required=False)
if zip_geo_raw is not None:
    assert_required_columns(zip_geo_raw, "zip_geo", ["zip_code", "latitude", "longitude", "created_at", "updated_at"])
    zip_geo = zip_geo_raw.select("zip_code", "latitude", "longitude", "created_at", "updated_at")
else:
    zip_geo = None

package_raw = read_silver_table("package")
assert_required_columns(
    package_raw,
    "package",
    [
        "package_id",
        "package_status_id",
        "service_type_id",
        "received_date",
        "contents",
        "weight_oz",
        "length_in",
        "width_in",
        "height_in",
        "created_at",
        "updated_at",
        "recipient_customer_id",
        "package_flow_type_id",
        "sender_customer_id",
        "sender_business_id",
    ],
)
package = package_raw.select(
    "package_id",
    "package_status_id",
    "service_type_id",
    "received_date",
    "contents",
    "weight_oz",
    "length_in",
    "width_in",
    "height_in",
    "created_at",
    "updated_at",
    "recipient_customer_id",
    "package_flow_type_id",
    "sender_customer_id",
    "sender_business_id",
)

package_status_raw = read_silver_table("package_status")
assert_required_columns(
    package_status_raw,
    "package_status",
    ["package_status_id", "status_name", "status_category", "sort_order", "is_final_status", "is_active", "allowed_service_type_id"],
)
package_status = package_status_raw.select(
    "package_status_id",
    "status_name",
    "status_category",
    "sort_order",
    "is_final_status",
    "is_active",
    "allowed_service_type_id",
)

package_flow_type_raw = read_silver_table("package_flow_type")
assert_required_columns(
    package_flow_type_raw,
    "package_flow_type",
    ["package_flow_type_id", "package_flow_type_name", "is_active"],
)
package_flow_type = package_flow_type_raw.select(
    "package_flow_type_id",
    "package_flow_type_name",
    "is_active",
)

service_type_raw = read_silver_table("service_type")
assert_required_columns(service_type_raw, "service_type", ["service_type_id", "service_type_name", "is_active"])
service_type = service_type_raw.select(
    "service_type_id",
    "service_type_name",
    "is_active",
)

shippingdetails_raw = read_silver_table("shippingdetails")
assert_required_columns(
    shippingdetails_raw,
    "shippingdetails",
    [
        "package_id",
        "recipient_address",
        "recipient_territory_id",
        "sender_address",
        "sender_territory_id",
        "estimated_delivery_distance",
        "recipient_first_name",
        "recipient_middle_initial",
        "recipient_last_name",
        "expected_delivery_date",
        "delivered_date",
        "created_at",
        "updated_at",
    ],
)
shippingdetails = shippingdetails_raw.select(
    "package_id",
    "recipient_address",
    "recipient_territory_id",
    "sender_address",
    "sender_territory_id",
    "estimated_delivery_distance",
    "recipient_first_name",
    "recipient_middle_initial",
    "recipient_last_name",
    "expected_delivery_date",
    "delivered_date",
    "created_at",
    "updated_at",
)

package_route_plan_raw = read_silver_table("package_route_plan", required=False)
if package_route_plan_raw is not None:
    assert_required_columns(
        package_route_plan_raw,
        "package_route_plan",
        [
            "package_id",
            "planned_origin_facility_id",
            "planned_destination_facility_id",
            "selection_source",
            "selected_at",
            "created_at",
            "updated_at",
        ],
    )
    package_route_plan = package_route_plan_raw.select(
        "package_id",
        "planned_origin_facility_id",
        "planned_destination_facility_id",
        "selection_source",
        "selected_at",
        "created_at",
        "updated_at",
    )
else:
    package_route_plan = None

package_movement_raw = read_silver_table("package_movement")
assert_required_columns(
    package_movement_raw,
    "package_movement",
    [
        "package_movement_id",
        "package_id",
        "package_movement_event_type_id",
        "package_status_id",
        "facility_id",
        "from_facility_id",
        "to_facility_id",
        "processed_by_employee_id",
        "event_timestamp",
        "movement_note",
        "created_at",
        "updated_at",
    ],
)
package_movement = package_movement_raw.select(
    "package_movement_id",
    "package_id",
    "package_movement_event_type_id",
    "package_status_id",
    "facility_id",
    "from_facility_id",
    "to_facility_id",
    "processed_by_employee_id",
    "event_timestamp",
    "movement_note",
    "created_at",
    "updated_at",
)

package_movement_event_type_raw = read_silver_table("package_movement_event_type")
assert_required_columns(
    package_movement_event_type_raw,
    "package_movement_event_type",
    [
        "package_movement_event_type_id",
        "event_type_name",
        "description",
        "default_package_status_name",
        "is_entry_event",
        "is_exit_event",
        "is_processing_event",
        "is_final_event",
        "sort_order",
        "is_active",
    ],
)
package_movement_event_type = package_movement_event_type_raw.select(
    "package_movement_event_type_id",
    "event_type_name",
    "description",
    "default_package_status_name",
    "is_entry_event",
    "is_exit_event",
    "is_processing_event",
    "is_final_event",
    "sort_order",
    "is_active",
)

shipping_cost_raw = read_silver_table("shipping_cost")
assert_required_columns(
    shipping_cost_raw,
    "shipping_cost",
    [
        "package_id",
        "actual_shipping_charge",
        "material_cost",
        "transportation_cost",
        "charge_source",
        "charge_recorded_at",
        "created_at",
        "updated_at",
    ],
)
shipping_cost = shipping_cost_raw.select(
    "package_id",
    "actual_shipping_charge",
    "material_cost",
    "transportation_cost",
    "charge_source",
    "charge_recorded_at",
    "created_at",
    "updated_at",
)

incident_raw = read_silver_table("incident")
assert_required_columns(
    incident_raw,
    "incident",
    [
        "incident_id",
        "reported_by_employee_id",
        "package_id",
        "incident_type_id",
        "incident_severity_id",
        "incident_status_id",
        "description",
        "incident_date",
        "created_at",
        "updated_at",
        "resolved_at",
        "resolved_by_employee_id",
        "resolution_note",
        "facility_id",
        "package_movement_id",
        "customer_id",
    ],
)
incident = incident_raw.select(
    "incident_id",
    "reported_by_employee_id",
    "package_id",
    "incident_type_id",
    "incident_severity_id",
    "incident_status_id",
    "description",
    "incident_date",
    "created_at",
    "updated_at",
    "resolved_at",
    "resolved_by_employee_id",
    "resolution_note",
    "facility_id",
    "package_movement_id",
    "customer_id",
)

incident_type_raw = read_silver_table("incident_type")
assert_required_columns(
    incident_type_raw,
    "incident_type",
    ["incident_type_id", "type_name", "type_category", "is_active", "blocks_package_movement"],
)
incident_type = incident_type_raw.select(
    "incident_type_id",
    "type_name",
    "type_category",
    "is_active",
    "blocks_package_movement",
)

incident_status_raw = read_silver_table("incident_status")
assert_required_columns(
    incident_status_raw,
    "incident_status",
    ["incident_status_id", "status_name", "sort_order", "is_closed_status", "is_active"],
)
incident_status = incident_status_raw.select(
    "incident_status_id",
    "status_name",
    "sort_order",
    "is_closed_status",
    "is_active",
)

incident_severity_raw = read_silver_table("incident_severity")
assert_required_columns(
    incident_severity_raw,
    "incident_severity",
    ["incident_severity_id", "severity_name", "sort_order", "is_active"],
)
incident_severity = incident_severity_raw.select(
    "incident_severity_id",
    "severity_name",
    "sort_order",
    "is_active",
)

refunds_raw = read_silver_table("refunds")
assert_required_columns(
    refunds_raw,
    "refunds",
    [
        "refund_id",
        "package_id",
        "incident_id",
        "refund_amount",
        "refund_reason",
        "refund_date",
        "refund_status",
        "created_at",
        "updated_at",
        "reviewed_at",
        "paid_at",
        "refund_note",
        "customer_id",
        "reviewed_by_employee_id",
    ],
)
refunds = refunds_raw.select(
    "refund_id",
    "package_id",
    "incident_id",
    "refund_amount",
    "refund_reason",
    "refund_date",
    "refund_status",
    "created_at",
    "updated_at",
    "reviewed_at",
    "paid_at",
    "refund_note",
    "customer_id",
    "reviewed_by_employee_id",
)

smartlocker_raw = read_silver_table("smartlocker")
assert_required_columns(
    smartlocker_raw,
    "smartlocker",
    ["locker_id", "locker_location_id", "locker_status", "created_at", "updated_at"],
)
smartlocker = smartlocker_raw.select(
    "locker_id",
    "locker_location_id",
    "locker_status",
    "created_at",
    "updated_at",
)

lockerlocations_raw = read_silver_table("lockerlocations", required=False)
if lockerlocations_raw is not None:
    assert_required_columns(lockerlocations_raw, "lockerlocations", ["locker_location_id", "location_name", "facility_id"])
    lockerlocations = lockerlocations_raw.select(
        "locker_location_id",
        "location_name",
        "facility_id",
    )
else:
    lockerlocations = None

package_to_locker_raw = read_silver_table("package_to_locker")
assert_required_columns(
    package_to_locker_raw,
    "package_to_locker",
    ["package_id", "locker_assignment_id", "assigned_at", "customer_id"],
)
package_to_locker = package_to_locker_raw.select(
    "package_id",
    "locker_assignment_id",
    "assigned_at",
    "customer_id",
)

lockerassignment_raw = read_silver_table("lockerassignment")
assert_required_columns(
    lockerassignment_raw,
    "lockerassignment",
    ["locker_assignment_id", "locker_id", "assigned_at", "expires_at", "retrieved_at", "customer_id"],
)
lockerassignment = lockerassignment_raw.select(
    "locker_assignment_id",
    "locker_id",
    "assigned_at",
    "expires_at",
    "retrieved_at",
    "customer_id",
)

In [0]:
# Source row-count and required-column checks

required_columns_by_table = {
    "customer": customer.columns,
    "business": business.columns,
    "facility": facility.columns,
    "facility_type": facility_type.columns,
    "departments": departments.columns,
    "department_type": department_type.columns,
    "employee": employee.columns,
    "territory": territory.columns,
    "package": package.columns,
    "package_status": package_status.columns,
    "package_flow_type": package_flow_type.columns,
    "service_type": service_type.columns,
    "shippingdetails": shippingdetails.columns,
    "package_movement": package_movement.columns,
    "package_movement_event_type": package_movement_event_type.columns,
    "shipping_cost": shipping_cost.columns,
    "incident": incident.columns,
    "incident_type": incident_type.columns,
    "incident_status": incident_status.columns,
    "incident_severity": incident_severity.columns,
    "refunds": refunds.columns,
    "smartlocker": smartlocker.columns,
    "package_to_locker": package_to_locker.columns,
    "lockerassignment": lockerassignment.columns,
}

loaded_tables = {
    "customer": customer,
    "business": business,
    "facility": facility,
    "facility_type": facility_type,
    "departments": departments,
    "department_type": department_type,
    "employee": employee,
    "territory": territory,
    "package": package,
    "package_status": package_status,
    "package_flow_type": package_flow_type,
    "service_type": service_type,
    "shippingdetails": shippingdetails,
    "package_movement": package_movement,
    "package_movement_event_type": package_movement_event_type,
    "shipping_cost": shipping_cost,
    "incident": incident,
    "incident_type": incident_type,
    "incident_status": incident_status,
    "incident_severity": incident_severity,
    "refunds": refunds,
    "smartlocker": smartlocker,
    "package_to_locker": package_to_locker,
    "lockerassignment": lockerassignment,
}

source_table_counts = {}
for table_name, table_df in loaded_tables.items():
    source_count = table_df.count()
    source_table_counts[table_name] = source_count
    record_validation(
        check_name="source_table_count",
        table_name=table_name,
        status="PASS",
        metric_value=source_count,
        details="source table loaded successfully",
    )

if zip_geo is None:
    record_warning("zip_geo is unavailable; dim_geography and facility/customer/business geography enrichment will omit latitude/longitude.")

if package_route_plan is None:
    record_warning("package_route_plan is unavailable; planned origin/destination facility enrichment will be null in package-related Gold tables.")

if lockerlocations is None:
    record_warning("lockerlocations is unavailable; smartlocker facility/geography enrichment will be null in dim_smartlocker and fact_smartlocker_assignment.")

display(
    spark.createDataFrame(validation_results).filter(F.col("check_name") == "source_table_count").orderBy("table_name")
)

check_name,details,metric_value,status,table_name
source_table_count,source table loaded successfully,3095,PASS,business
source_table_count,source table loaded successfully,96096,PASS,customer
source_table_count,source table loaded successfully,6,PASS,department_type
source_table_count,source table loaded successfully,1596,PASS,departments
source_table_count,source table loaded successfully,5000,PASS,employee
source_table_count,source table loaded successfully,541,PASS,facility
source_table_count,source table loaded successfully,5,PASS,facility_type
source_table_count,source table loaded successfully,6286,PASS,incident
source_table_count,source table loaded successfully,4,PASS,incident_severity
source_table_count,source table loaded successfully,4,PASS,incident_status


In [0]:
# Build dim_date

date_event_frames = [
    package.select(F.to_date("received_date").alias("event_date")),
    package.select(F.to_date("created_at").alias("event_date")),
    package.select(F.to_date("updated_at").alias("event_date")),
    package_movement.select(F.to_date("event_timestamp").alias("event_date")),
    shippingdetails.select(F.to_date("expected_delivery_date").alias("event_date")),
    shippingdetails.select(F.to_date("delivered_date").alias("event_date")),
    shipping_cost.select(F.to_date("charge_recorded_at").alias("event_date")),
    incident.select(F.to_date("incident_date").alias("event_date")),
    incident.select(F.to_date("resolved_at").alias("event_date")),
    refunds.select(F.to_date("refund_date").alias("event_date")),
    refunds.select(F.to_date("reviewed_at").alias("event_date")),
    refunds.select(F.to_date("paid_at").alias("event_date")),
    package_to_locker.select(F.to_date("assigned_at").alias("event_date")),
    lockerassignment.select(F.to_date("assigned_at").alias("event_date")),
    lockerassignment.select(F.to_date("expires_at").alias("event_date")),
    lockerassignment.select(F.to_date("retrieved_at").alias("event_date")),
]

all_event_dates = reduce(lambda left, right: left.unionByName(right), date_event_frames).filter(F.col("event_date").isNotNull())
date_bounds = all_event_dates.agg(F.min("event_date").alias("min_date"), F.max("event_date").alias("max_date")).first()

if date_bounds["min_date"] is None or date_bounds["max_date"] is None:
    # TODO: If the runtime source ever lacks date coverage entirely, replace this fallback with a project-approved static window.
    min_date_literal = F.to_date(F.lit("2015-01-01"))
    max_date_literal = F.to_date(F.lit("2030-12-31"))
else:
    min_date_literal = F.lit(date_bounds["min_date"])
    max_date_literal = F.lit(date_bounds["max_date"])

dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(min_date_literal, max_date_literal, F.expr("interval 1 day"))).alias("full_date"))
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("full_date"),
        F.date_format("full_date", "EEEE").alias("day_of_week"),
        F.dayofweek("full_date").alias("day_of_week_number"),
        F.dayofmonth("full_date").alias("day_of_month"),
        F.dayofyear("full_date").alias("day_of_year"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter_number"),
        F.year("full_date").alias("year_number"),
        F.when(F.dayofweek("full_date").isin([1, 7]), F.lit(True)).otherwise(F.lit(False)).alias("is_weekend"),
        F.when(F.dayofmonth("full_date") == 1, F.lit(True)).otherwise(F.lit(False)).alias("is_month_start"),
        F.when(F.last_day("full_date") == F.col("full_date"), F.lit(True)).otherwise(F.lit(False)).alias("is_month_end"),
    )
)

dim_date = with_gold_audit_columns(dim_date)
dim_date = write_gold_table(dim_date, "dim_date")
add_table_row_count("dim_date", dim_date)

In [0]:
# Build lookup dimensions

dim_service_type = with_gold_audit_columns(
    service_type.select(
        deterministic_key(normalized_str(F.col("service_type_id"))).alias("service_type_key"),
        F.col("service_type_id").alias("service_type_id_source"),
        "service_type_name",
        "is_active",
    )
)
dim_service_type = write_gold_table(dim_service_type, "dim_service_type")
add_table_row_count("dim_service_type", dim_service_type)

dim_package_status = with_gold_audit_columns(
    package_status.select(
        deterministic_key(normalized_str(F.col("package_status_id"))).alias("package_status_key"),
        F.col("package_status_id").alias("package_status_id_source"),
        "status_name",
        "status_category",
        "sort_order",
        "is_final_status",
        "is_active",
        "allowed_service_type_id",
    )
)
dim_package_status = write_gold_table(dim_package_status, "dim_package_status")
add_table_row_count("dim_package_status", dim_package_status)

dim_package_flow_type = with_gold_audit_columns(
    package_flow_type.select(
        deterministic_key(normalized_str(F.col("package_flow_type_id"))).alias("package_flow_type_key"),
        F.col("package_flow_type_id").alias("package_flow_type_id_source"),
        "package_flow_type_name",
        "is_active",
        safe_bool_from_name("package_flow_type_name", ["B2C"]).alias("is_b2c"),
        safe_bool_from_name("package_flow_type_name", ["P2P"]).alias("is_p2p"),
    )
)
dim_package_flow_type = write_gold_table(dim_package_flow_type, "dim_package_flow_type")
add_table_row_count("dim_package_flow_type", dim_package_flow_type)

dim_incident_type = with_gold_audit_columns(
    incident_type.select(
        deterministic_key(normalized_str(F.col("incident_type_id"))).alias("incident_type_key"),
        F.col("incident_type_id").alias("incident_type_id_source"),
        F.col("type_name").alias("incident_type_name"),
        "type_category",
        "is_active",
        "blocks_package_movement",
    )
)
dim_incident_type = write_gold_table(dim_incident_type, "dim_incident_type")
add_table_row_count("dim_incident_type", dim_incident_type)

dim_refund_status = with_gold_audit_columns(
    refunds.select(F.trim(F.col("refund_status")).alias("refund_status_name"))
    .filter(F.col("refund_status_name").isNotNull() & (F.col("refund_status_name") != ""))
    .distinct()
    .select(
        deterministic_key(normalized_str(F.col("refund_status_name"))).alias("refund_status_key"),
        F.col("refund_status_name").alias("refund_status_id_source"),
        "refund_status_name",
        safe_bool_from_name("refund_status_name", ["Pending"]).alias("is_pending"),
        safe_bool_from_name("refund_status_name", ["Approved"]).alias("is_approved"),
        safe_bool_from_name("refund_status_name", ["Rejected"]).alias("is_rejected"),
        safe_bool_from_name("refund_status_name", ["Paid"]).alias("is_paid"),
        safe_bool_from_name("refund_status_name", ["Cancelled"]).alias("is_cancelled"),
    )
)
dim_refund_status = write_gold_table(dim_refund_status, "dim_refund_status")
add_table_row_count("dim_refund_status", dim_refund_status)

# Build package movement event type dimension

dim_package_movement_event_type = with_gold_audit_columns(
    package_movement_event_type.select(
        deterministic_key(
            normalized_str(F.col("package_movement_event_type_id"))
        ).alias("package_movement_event_type_key"),

        F.col("package_movement_event_type_id").alias("package_movement_event_type_id_source"),
        F.col("event_type_name"),
        F.col("description").alias("event_type_description"),
        F.col("default_package_status_name"),

        bool_from_tinyint(F.col("is_entry_event")).alias("is_entry_event"),
        bool_from_tinyint(F.col("is_exit_event")).alias("is_exit_event"),
        bool_from_tinyint(F.col("is_processing_event")).alias("is_processing_event"),
        bool_from_tinyint(F.col("is_final_event")).alias("is_final_event"),

        F.col("sort_order"),
        F.col("is_active"),
    )
)

dim_package_movement_event_type = write_gold_table(
    dim_package_movement_event_type,
    "dim_package_movement_event_type"
)

add_table_row_count(
    "dim_package_movement_event_type",
    dim_package_movement_event_type
)

# Build incident status dimension

dim_incident_status = with_gold_audit_columns(
    incident_status.select(
        deterministic_key(
            normalized_str(F.col("incident_status_id"))
        ).alias("incident_status_key"),

        F.col("incident_status_id").alias("incident_status_id_source"),
        F.col("status_name").alias("incident_status_name"),

        safe_bool_from_name("status_name", ["Open"]).alias("is_open_status"),
        safe_bool_from_name("status_name", ["Investigating"]).alias("is_investigating_status"),
        safe_bool_from_name("status_name", ["Resolved"]).alias("is_resolved_status"),
        safe_bool_from_name("status_name", ["Closed"]).alias("is_closed_status_name"),

        bool_from_tinyint(F.col("is_closed_status")).alias("is_closed_status"),
        F.col("sort_order"),
        F.col("is_active"),
    )
)

dim_incident_status = write_gold_table(
    dim_incident_status,
    "dim_incident_status"
)

add_table_row_count(
    "dim_incident_status",
    dim_incident_status
)

# Build incident severity dimension

dim_incident_severity = with_gold_audit_columns(
    incident_severity.select(
        deterministic_key(
            normalized_str(F.col("incident_severity_id"))
        ).alias("incident_severity_key"),

        F.col("incident_severity_id").alias("incident_severity_id_source"),
        F.col("severity_name").alias("incident_severity_name"),

        safe_bool_from_name("severity_name", ["Low"]).alias("is_low_severity"),
        safe_bool_from_name("severity_name", ["Medium"]).alias("is_medium_severity"),
        safe_bool_from_name("severity_name", ["High"]).alias("is_high_severity"),
        safe_bool_from_name("severity_name", ["Critical"]).alias("is_critical_severity"),

        F.when(
            F.lower(F.coalesce(F.col("severity_name"), F.lit(""))).isin(["high", "critical"]),
            F.lit(True)
        ).otherwise(F.lit(False)).alias("is_escalation_severity"),

        F.col("sort_order"),
        F.col("is_active"),
    )
)

dim_incident_severity = write_gold_table(
    dim_incident_severity,
    "dim_incident_severity"
)

add_table_row_count(
    "dim_incident_severity",
    dim_incident_severity
)

# Build smartlocker assignment status dimension

smartlocker_assignment_status_schema = T.StructType([
    T.StructField("assignment_status_name", T.StringType(), False),
    T.StructField("assignment_status_display_name", T.StringType(), False),
    T.StructField("sort_order", T.IntegerType(), False),
    T.StructField("is_assigned_status", T.BooleanType(), False),
    T.StructField("is_pending_pickup_status", T.BooleanType(), False),
    T.StructField("is_picked_up_status", T.BooleanType(), False),
    T.StructField("is_final_assignment_status", T.BooleanType(), False),
])

smartlocker_assignment_status_seed = spark.createDataFrame(
    [
        ("assigned", "Assigned", 1, True, False, False, False),
        ("awaiting_pickup", "Awaiting Pickup", 2, True, True, False, False),
        ("picked_up", "Picked Up", 3, True, False, True, True),
    ],
    schema=smartlocker_assignment_status_schema,
)

dim_smartlocker_assignment_status = with_gold_audit_columns(
    smartlocker_assignment_status_seed.select(
        deterministic_key(
            normalized_str(F.col("assignment_status_name"))
        ).alias("assignment_status_key"),

        F.col("assignment_status_name"),
        F.col("assignment_status_display_name"),
        F.col("sort_order"),
        F.col("is_assigned_status"),
        F.col("is_pending_pickup_status"),
        F.col("is_picked_up_status"),
        F.col("is_final_assignment_status"),
    )
)

dim_smartlocker_assignment_status = write_gold_table(
    dim_smartlocker_assignment_status,
    "dim_smartlocker_assignment_status"
)

add_table_row_count(
    "dim_smartlocker_assignment_status",
    dim_smartlocker_assignment_status
)

In [0]:
# Build geography, facility, department, employee, and smartlocker dimensions

territory_base = territory.alias("t")
if zip_geo is not None:
    territory_base = territory_base.join(
        zip_geo.alias("zg"),
        F.col("t.zip_code") == F.col("zg.zip_code"),
        "left",
    )
    geography_select = [
        F.col("t.territory_id"),
        F.col("t.territory_id").alias("territory_id_source"),
        F.col("t.zip_code"),
        F.col("t.city").alias("territory_city"),
        F.col("t.county").alias("territory_county"),
        F.col("t.state").alias("territory_state"),
        F.col("zg.latitude").alias("latitude"),
        F.col("zg.longitude").alias("longitude"),
        F.col("t.created_at"),
        F.col("t.updated_at"),
    ]
else:
    geography_select = [
        F.col("t.territory_id"),
        F.col("t.territory_id").alias("territory_id_source"),
        F.col("t.zip_code"),
        F.col("t.city").alias("territory_city"),
        F.col("t.county").alias("territory_county"),
        F.col("t.state").alias("territory_state"),
        F.lit(None).cast("decimal(38,18)").alias("latitude"),
        F.lit(None).cast("decimal(38,18)").alias("longitude"),
        F.col("t.created_at"),
        F.col("t.updated_at"),
    ]

dim_geography = with_gold_audit_columns(
    territory_base.select(
        deterministic_key(normalized_str(F.col("t.territory_id"))).alias("geography_key"),
        *geography_select,
    )
)
dim_geography = write_gold_table(dim_geography, "dim_geography")
add_table_row_count("dim_geography", dim_geography)

facility_geo = dim_geography.select(
    "territory_id_source",
    "geography_key",
    "zip_code",
    "territory_city",
    "territory_county",
    "territory_state",
    "latitude",
    "longitude",
)

dim_facility = with_gold_audit_columns(
    facility.alias("f")
    .join(facility_type.alias("ft"), F.col("f.facility_type_id") == F.col("ft.facility_type_id"), "left")
    .join(facility_geo.alias("g"), F.col("f.territory_id") == F.col("g.territory_id_source"), "left")
    .select(
        deterministic_key(normalized_str(F.col("f.facility_id"))).alias("facility_key"),
        F.col("f.facility_id").alias("facility_id_source"),
        F.col("f.facility_name"),
        F.col("f.facility_type_id").alias("facility_type_id_source"),
        F.col("ft.facility_type_code"),
        F.col("ft.facility_type_name"),
        F.col("ft.description").alias("facility_type_description"),
        F.col("ft.is_customer_facing"),
        F.col("ft.handles_retail"),
        F.col("ft.handles_processing"),
        F.col("ft.handles_distribution"),
        F.col("ft.handles_local_delivery"),
        F.col("f.manager_employee_id").alias("manager_employee_id_source"),
        F.col("f.street_address"),
        F.col("f.county"),
        F.col("f.city"),
        F.col("f.state_code"),
        F.col("f.zip_code"),
        F.col("f.facility_department_prefix"),
        F.col("f.territory_id").alias("territory_id_source"),
        F.col("g.geography_key"),
        F.col("g.latitude"),
        F.col("g.longitude"),
        F.when(F.lower(F.col("ft.facility_type_name")) == "post office", F.lit(True)).otherwise(F.lit(False)).alias("is_post_office"),
        F.when(F.lower(F.col("ft.facility_type_name")) == "network facilities", F.lit(True)).otherwise(F.lit(False)).alias("is_network_facility"),
        F.when(F.lower(F.col("ft.facility_type_name")) == "mail processing", F.lit(True)).otherwise(F.lit(False)).alias("is_mail_processing"),
        F.when(F.lower(F.col("ft.facility_type_name")) == "vehicle maintenance", F.lit(True)).otherwise(F.lit(False)).alias("is_vehicle_maintenance"),
        F.when(F.lower(F.col("ft.facility_type_name")) == "administrative office", F.lit(True)).otherwise(F.lit(False)).alias("is_administrative_office"),
    )
)
dim_facility = write_gold_table(dim_facility, "dim_facility")
add_table_row_count("dim_facility", dim_facility)

dim_department = with_gold_audit_columns(
    departments.alias("d")
    .join(department_type.alias("dt"), F.col("d.department_type_id") == F.col("dt.department_type_id"), "left")
    .join(dim_facility.alias("f"), F.col("d.facility_id") == F.col("f.facility_id_source"), "left")
    .select(
        deterministic_key(normalized_str(F.col("d.department_id"))).alias("department_key"),
        F.col("d.department_id").alias("department_id_source"),
        F.col("d.department_name"),
        F.col("d.department_type_id").alias("department_type_id_source"),
        F.col("dt.department_type_code"),
        F.col("dt.department_type_name"),
        F.col("dt.description").alias("department_type_description"),
        F.col("d.manager_employee_id").alias("manager_employee_id_source"),
        F.col("d.manager_start_date"),
        F.col("f.facility_key"),
        F.col("f.facility_id_source"),
        F.col("f.facility_name"),
        F.col("f.facility_type_name"),
        F.col("f.geography_key"),
        F.col("f.city").alias("facility_city"),
        F.col("f.county").alias("facility_county"),
        F.col("f.state_code").alias("facility_state_code"),
        F.col("f.zip_code").alias("facility_zip_code"),
        F.col("f.territory_id_source").alias("territory_id_source"),
    )
)
dim_department = write_gold_table(dim_department, "dim_department")
add_table_row_count("dim_department", dim_department)

dim_employee = with_gold_audit_columns(
    employee.alias("e")
    .join(dim_department.alias("d"), F.col("e.department_id") == F.col("d.department_id_source"), "left")
    .select(
        deterministic_key(normalized_str(F.col("e.employee_id"))).alias("employee_key"),
        F.col("e.employee_id").alias("employee_id_source"),
        F.col("e.full_name").alias("employee_name"),
        F.col("e.phone_number"),
        F.col("e.email"),
        F.col("e.street_address"),
        F.col("e.job_title"),
        F.col("e.salary"),
        F.col("e.hours_worked"),
        F.col("e.manager_employee_id").alias("manager_employee_id_source"),
        F.col("e.user_id").alias("user_id_source"),
        F.col("d.department_key"),
        F.col("d.department_id_source"),
        F.col("d.department_name"),
        F.col("d.department_type_id_source"),
        F.col("d.department_type_name"),
        F.col("d.facility_key"),
        F.col("d.facility_id_source"),
        F.col("d.facility_name"),
        F.col("d.facility_type_name"),
        F.col("d.geography_key"),
        F.col("d.facility_city"),
        F.col("d.facility_county"),
        F.col("d.facility_state_code"),
        F.col("d.facility_zip_code"),
        F.col("d.territory_id_source"),
    )
)
dim_employee = write_gold_table(dim_employee, "dim_employee")
add_table_row_count("dim_employee", dim_employee)

if lockerlocations is not None:
    locker_location_enriched = lockerlocations.alias("ll").join(
        dim_facility.alias("f"),
        F.col("ll.facility_id") == F.col("f.facility_id_source"),
        "left",
    )
else:
    locker_location_enriched = None

if locker_location_enriched is not None:
    dim_smartlocker = with_gold_audit_columns(
        smartlocker.alias("sl")
        .join(locker_location_enriched.alias("ll"), F.col("sl.locker_location_id") == F.col("ll.locker_location_id"), "left")
        .select(
            deterministic_key(normalized_str(F.col("sl.locker_id"))).alias("smartlocker_key"),
            F.col("sl.locker_id").alias("locker_id_source"),
            F.col("sl.locker_location_id").alias("locker_location_id_source"),
            F.col("ll.location_name").alias("locker_location_name"),
            F.col("sl.locker_status"),
            F.col("ll.facility_key"),
            F.col("ll.facility_id_source"),
            F.col("ll.facility_name"),
            F.col("ll.geography_key"),
            F.col("ll.city").alias("facility_city"),
            F.col("ll.county").alias("facility_county"),
            F.col("ll.state_code").alias("facility_state_code"),
            F.col("ll.zip_code").alias("facility_zip_code"),
        )
    )
else:
    dim_smartlocker = with_gold_audit_columns(
        smartlocker.select(
            deterministic_key(normalized_str(F.col("locker_id"))).alias("smartlocker_key"),
            F.col("locker_id").alias("locker_id_source"),
            F.col("locker_location_id").alias("locker_location_id_source"),
            F.lit(None).cast("string").alias("locker_location_name"),
            F.col("locker_status"),
            F.lit(None).cast("long").alias("facility_key"),
            F.lit(None).cast("int").alias("facility_id_source"),
            F.lit(None).cast("string").alias("facility_name"),
            F.lit(None).cast("long").alias("geography_key"),
            F.lit(None).cast("string").alias("facility_city"),
            F.lit(None).cast("string").alias("facility_county"),
            F.lit(None).cast("string").alias("facility_state_code"),
            F.lit(None).cast("string").alias("facility_zip_code"),
        )
    )

dim_smartlocker = write_gold_table(dim_smartlocker, "dim_smartlocker")
add_table_row_count("dim_smartlocker", dim_smartlocker)

In [0]:
# Build customer, business, and package dimensions

dim_customer = with_gold_audit_columns(
    customer.alias("c")
    .join(dim_geography.alias("g"), F.col("c.territory_id") == F.col("g.territory_id_source"), "left")
    .join(dim_facility.alias("f"), F.col("c.preferred_facility_id") == F.col("f.facility_id_source"), "left")
    .select(
        deterministic_key(normalized_binary(F.col("c.customer_id"))).alias("customer_key"),
        F.col("c.customer_id").alias("customer_id_source"),
        # Explicit aliasing avoids ambiguous references after geography/facility joins.
        F.col("c.first_name").alias("first_name"),
        F.col("c.middle_initial").alias("middle_initial"),
        F.col("c.last_name").alias("last_name"),
        F.col("c.street_address").alias("street_address"),
        F.col("c.county").alias("county"),
        F.col("c.city").alias("city"),
        F.col("c.state_code").alias("state_code"),
        F.col("c.zip_code").alias("zip_code"),
        F.col("c.territory_id").alias("territory_id_source"),
        F.col("g.geography_key"),
        F.col("c.phone_number").alias("phone_number"),
        F.col("c.email").alias("email"),
        F.col("c.email_address").alias("email_address"),
        F.col("c.user_id").alias("user_id_source"),
        F.col("c.preferred_facility_id").alias("preferred_facility_id_source"),
        F.col("f.facility_key").alias("preferred_facility_key"),
        F.col("f.facility_name").alias("preferred_facility_name"),
        F.col("c.birth_date").alias("birth_date"),
        F.when(F.col("c.birth_date").isNull(), F.lit(None).cast("int")).otherwise(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12)).alias("current_age_years"),
        F.when(F.col("c.birth_date").isNull(), F.lit("Unknown"))
        .when(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12) < 25, F.lit("18-24"))
        .when(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12) < 35, F.lit("25-34"))
        .when(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12) < 45, F.lit("35-44"))
        .when(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12) < 55, F.lit("45-54"))
        .when(F.floor(F.months_between(F.current_date(), F.col("c.birth_date")) / 12) < 65, F.lit("55-64"))
        .otherwise(F.lit("65+")).alias("age_band"),
        F.col("c.marital_status").alias("marital_status"),
        F.col("c.gender").alias("gender"),
        F.col("c.annual_income").alias("annual_income"),
        F.when(F.col("c.annual_income").isNull(), F.lit("Unknown"))
        .when(F.col("c.annual_income") >= F.lit(150000), F.lit("Very High"))
        .when(F.col("c.annual_income") >= F.lit(100000), F.lit("High"))
        .when(F.col("c.annual_income") >= F.lit(50000), F.lit("Medium"))
        .otherwise(F.lit("Low")).alias("annual_income_band"),
        F.col("c.total_children").alias("total_children"),
        F.col("c.education_level").alias("education_level"),
        F.col("c.occupation").alias("occupation"),
        F.col("c.home_owner").alias("home_owner"),
        F.when(F.lower(F.coalesce(F.col("c.home_owner"), F.lit("n"))) == "y", F.lit(True)).otherwise(F.lit(False)).alias("home_owner_flag"),
    )
)
dim_customer = write_gold_table(dim_customer, "dim_customer")
add_table_row_count("dim_customer", dim_customer)

dim_business = with_gold_audit_columns(
    business.alias("b")
    .join(dim_geography.alias("g"), F.col("b.territory_id") == F.col("g.territory_id_source"), "left")
    .join(dim_facility.alias("f"), F.col("b.preferred_facility_id") == F.col("f.facility_id_source"), "left")
    .select(
        deterministic_key(normalized_binary(F.col("b.business_id"))).alias("business_key"),
        F.col("b.business_id").alias("business_id_source"),
        # Explicit aliasing avoids ambiguous references after geography/facility joins.
        F.col("b.business_name").alias("business_name"),
        F.col("b.street_address").alias("street_address"),
        F.col("b.county").alias("county"),
        F.col("b.city").alias("city"),
        F.col("b.state_code").alias("state_code"),
        F.col("b.zip_code").alias("zip_code"),
        F.col("b.territory_id").alias("territory_id_source"),
        F.col("g.geography_key"),
        F.col("b.phone_number").alias("phone_number"),
        F.col("b.email").alias("email"),
        F.col("b.preferred_facility_id").alias("preferred_facility_id_source"),
        F.col("f.facility_key").alias("preferred_facility_key"),
        F.col("f.facility_name").alias("preferred_facility_name"),
    )
)
dim_business = write_gold_table(dim_business, "dim_business")
add_table_row_count("dim_business", dim_business)

package_route_plan_enriched = package_route_plan.alias("prp") if package_route_plan is not None else None

package_join = (
    package.alias("p")
    .join(package_status.alias("ps"), F.col("p.package_status_id") == F.col("ps.package_status_id"), "left")
    .join(service_type.alias("st"), F.col("p.service_type_id") == F.col("st.service_type_id"), "left")
    .join(package_flow_type.alias("pft"), F.col("p.package_flow_type_id") == F.col("pft.package_flow_type_id"), "left")
    .join(shippingdetails.alias("sd"), F.col("p.package_id") == F.col("sd.package_id"), "left")
)

if package_route_plan_enriched is not None:
    package_join = package_join.join(package_route_plan_enriched, F.col("p.package_id") == F.col("prp.package_id"), "left")

package_select_columns = [
    deterministic_key(normalized_binary(F.col("p.package_id"))).alias("package_key"),
    F.col("p.package_id").alias("package_id_source"),
    F.col("p.package_status_id").alias("package_status_id_source"),
    deterministic_key(normalized_str(F.col("p.package_status_id"))).alias("package_status_key"),
    F.col("ps.status_name").alias("package_status_name"),
    F.col("ps.status_category").alias("package_status_category"),
    F.col("ps.is_final_status"),
    F.col("p.service_type_id").alias("service_type_id_source"),
    deterministic_key(normalized_str(F.col("p.service_type_id"))).alias("service_type_key"),
    F.col("p.package_flow_type_id").alias("package_flow_type_id_source"),
    deterministic_key(normalized_str(F.col("p.package_flow_type_id"))).alias("package_flow_type_key"),
    F.col("pft.package_flow_type_name"),
    F.when(F.lower(F.col("pft.package_flow_type_name")) == "b2c", F.lit(True)).otherwise(F.lit(False)).alias("is_b2c"),
    F.when(F.lower(F.col("pft.package_flow_type_name")) == "p2p", F.lit(True)).otherwise(F.lit(False)).alias("is_p2p"),
    F.col("p.recipient_customer_id").alias("recipient_customer_id_source"),
    F.col("p.sender_customer_id").alias("sender_customer_id_source"),
    F.col("p.sender_business_id").alias("sender_business_id_source"),
    F.col("p.received_date"),
    F.col("p.contents"),
    F.col("p.weight_oz"),
    F.col("p.length_in"),
    F.col("p.width_in"),
    F.col("p.height_in"),
    (F.col("p.length_in") * F.col("p.width_in") * F.col("p.height_in")).alias("package_volume_cubic_in"),
    F.col("sd.sender_address"),
    F.col("sd.sender_territory_id").alias("sender_territory_id_source"),
    F.col("sd.recipient_address"),
    F.col("sd.recipient_territory_id").alias("recipient_territory_id_source"),
    F.col("sd.recipient_first_name"),
    F.col("sd.recipient_middle_initial"),
    F.col("sd.recipient_last_name"),
    F.col("sd.estimated_delivery_distance"),
    F.col("sd.expected_delivery_date"),
    F.col("sd.delivered_date"),
]

if package_route_plan_enriched is not None:
    package_select_columns.extend(
        [
            F.col("prp.planned_origin_facility_id").alias("planned_origin_facility_id_source"),
            F.col("prp.planned_destination_facility_id").alias("planned_destination_facility_id_source"),
            F.col("prp.selection_source").alias("route_selection_source"),
            F.col("prp.selected_at").alias("route_selected_at"),
            deterministic_key(normalized_str(F.col("prp.planned_origin_facility_id"))).alias("planned_origin_facility_key"),
            deterministic_key(normalized_str(F.col("prp.planned_destination_facility_id"))).alias("planned_destination_facility_key"),
        ]
    )
else:
    package_select_columns.extend(
        [
            F.lit(None).cast("int").alias("planned_origin_facility_id_source"),
            F.lit(None).cast("int").alias("planned_destination_facility_id_source"),
            F.lit(None).cast("string").alias("route_selection_source"),
            F.lit(None).cast("timestamp").alias("route_selected_at"),
            F.lit(None).cast("long").alias("planned_origin_facility_key"),
            F.lit(None).cast("long").alias("planned_destination_facility_key"),
        ]
    )

dim_package = with_gold_audit_columns(package_join.select(*package_select_columns))
dim_package = write_gold_table(dim_package, "dim_package")
add_table_row_count("dim_package", dim_package)

In [0]:
# Dimension validation

dimension_specs = {
    "dim_date": (dim_date, ["date_key"], []),
    "dim_customer": (dim_customer, ["customer_key"], ["customer_id_source"]),
    "dim_business": (dim_business, ["business_key"], ["business_id_source"]),
    "dim_package": (dim_package, ["package_key"], ["package_id_source"]),
    "dim_facility": (dim_facility, ["facility_key"], ["facility_id_source"]),
    "dim_department": (dim_department, ["department_key"], ["department_id_source"]),
    "dim_employee": (dim_employee, ["employee_key"], ["employee_id_source"]),
    "dim_geography": (dim_geography, ["geography_key"], ["territory_id_source"]),
    "dim_service_type": (dim_service_type, ["service_type_key"], ["service_type_id_source"]),
    "dim_package_status": (dim_package_status, ["package_status_key"], ["package_status_id_source"]),
    "dim_package_flow_type": (dim_package_flow_type, ["package_flow_type_key"], ["package_flow_type_id_source"]),
    "dim_incident_type": (dim_incident_type, ["incident_type_key"], ["incident_type_id_source"]),
    "dim_refund_status": (dim_refund_status, ["refund_status_key"], ["refund_status_id_source"]),
    "dim_smartlocker": (dim_smartlocker, ["smartlocker_key"], ["locker_id_source"]),

    "dim_package_movement_event_type": (
        dim_package_movement_event_type,
        ["package_movement_event_type_key"],
        ["package_movement_event_type_id_source"]
    ),

    "dim_incident_status": (
        dim_incident_status,
        ["incident_status_key"],
        ["incident_status_id_source"]
    ),

    "dim_incident_severity": (
        dim_incident_severity,
        ["incident_severity_key"],
        ["incident_severity_id_source"]
    ),

    "dim_smartlocker_assignment_status": (
        dim_smartlocker_assignment_status,
        ["assignment_status_key"],
        ["assignment_status_name"]
    ),
}

for dim_table_name, (dim_df, surrogate_keys, natural_keys) in dimension_specs.items():
    for surrogate_key in surrogate_keys:
        validate_non_null(dim_df, dim_table_name, surrogate_key)
        validate_unique_keys(dim_df, dim_table_name, [surrogate_key])
    for natural_key in natural_keys:
        validate_unique_keys(dim_df, dim_table_name, [natural_key], check_name="duplicate_source_grain_check")

display(spark.createDataFrame(validation_results).orderBy("table_name", "check_name"))

check_name,details,metric_value,status,table_name
source_table_count,source table loaded successfully,3095,PASS,business
source_table_count,source table loaded successfully,96096,PASS,customer
source_table_count,source table loaded successfully,6,PASS,department_type
source_table_count,source table loaded successfully,1596,PASS,departments
duplicate_key_check,columns=business_key,0,PASS,dim_business
duplicate_source_grain_check,columns=business_id_source,0,PASS,dim_business
non_null_key_check,column=business_key,0,PASS,dim_business
duplicate_key_check,columns=customer_key,0,PASS,dim_customer
duplicate_source_grain_check,columns=customer_id_source,0,PASS,dim_customer
non_null_key_check,column=customer_key,0,PASS,dim_customer


In [0]:
# Build fact_package

movement_enriched_for_package = (
    package_movement.alias("pm")
    .join(
        package_movement_event_type.alias("pmet"),
        F.col("pm.package_movement_event_type_id") == F.col("pmet.package_movement_event_type_id"),
        "left",
    )
    .join(
        package_status.alias("ps"),
        F.col("pm.package_status_id") == F.col("ps.package_status_id"),
        "left",
    )
)

movement_package_agg = movement_enriched_for_package.groupBy(F.col("pm.package_id").alias("package_id")).agg(
    F.count("*").alias("movement_count"),
    F.min("pm.event_timestamp").alias("first_movement_timestamp"),
    F.max("pm.event_timestamp").alias("last_movement_timestamp"),
    F.min(
        F.when(F.lower(F.col("pmet.event_type_name")).like("%sort%"), F.col("pm.event_timestamp"))
    ).alias("sorted_at_origin_timestamp"),
    F.min(
        F.when(F.lower(F.col("pmet.event_type_name")).like("%depart%"), F.col("pm.event_timestamp"))
    ).alias("departed_origin_timestamp"),
    F.max(
        F.when(F.lower(F.trim(F.coalesce(F.col("ps.status_name"), F.lit("")))) == "delivered", F.col("pm.event_timestamp"))
    ).alias("delivered_movement_timestamp"),
)

incident_package_agg = incident.groupBy("package_id").agg(
    F.count("*").alias("incident_count"),
    F.sum(F.when(F.col("resolved_at").isNotNull(), F.lit(1)).otherwise(F.lit(0))).alias("resolved_incident_count"),
)

refund_package_agg = refunds.groupBy("package_id").agg(
    F.count("*").alias("refund_count"),
    F.sum("refund_amount").alias("refund_amount_total"),
)

shipping_cost_package_agg = shipping_cost.groupBy("package_id").agg(
    F.sum("actual_shipping_charge").alias("shipping_revenue_amount"),
    F.sum("material_cost").alias("material_cost_amount"),
    F.sum("transportation_cost").alias("transportation_cost_amount"),
    F.min("charge_recorded_at").alias("first_charge_recorded_at"),
    F.max("charge_recorded_at").alias("last_charge_recorded_at"),
)

# Controlled status normalization prevents fuzzy matches such as "Out for Delivery" from counting as delivered.
status_name_norm = F.lower(F.trim(F.coalesce(F.col("ps.status_name"), F.lit(""))))
status_category_norm = F.lower(F.trim(F.coalesce(F.col("ps.status_category"), F.lit(""))))

fact_package = with_gold_audit_columns(
    package.alias("p")
    .join(dim_package.alias("dp"), F.col("p.package_id") == F.col("dp.package_id_source"), "left")
    .join(dim_customer.alias("recipient"), F.col("p.recipient_customer_id") == F.col("recipient.customer_id_source"), "left")
    .join(dim_customer.alias("sender_customer"), F.col("p.sender_customer_id") == F.col("sender_customer.customer_id_source"), "left")
    .join(dim_business.alias("sender_business"), F.col("p.sender_business_id") == F.col("sender_business.business_id_source"), "left")
    .join(movement_package_agg.alias("m"), F.col("p.package_id") == F.col("m.package_id"), "left")
    .join(incident_package_agg.alias("i"), F.col("p.package_id") == F.col("i.package_id"), "left")
    .join(refund_package_agg.alias("r"), F.col("p.package_id") == F.col("r.package_id"), "left")
    .join(shipping_cost_package_agg.alias("sc"), F.col("p.package_id") == F.col("sc.package_id"), "left")
    .join(shippingdetails.alias("sd"), F.col("p.package_id") == F.col("sd.package_id"), "left")
    .join(package_status.alias("ps"), F.col("p.package_status_id") == F.col("ps.package_status_id"), "left")
    .join(package_flow_type.alias("pft"), F.col("p.package_flow_type_id") == F.col("pft.package_flow_type_id"), "left")
    .join(service_type.alias("st"), F.col("p.service_type_id") == F.col("st.service_type_id"), "left")
    .join(dim_geography.alias("recipient_geo"), F.col("sd.recipient_territory_id") == F.col("recipient_geo.territory_id_source"), "left")
    .join(dim_geography.alias("sender_geo"), F.col("sd.sender_territory_id") == F.col("sender_geo.territory_id_source"), "left")
    .select(
        deterministic_key(normalized_binary(F.col("p.package_id"))).alias("fact_package_key"),
        F.col("dp.package_key"),
        F.col("p.package_id").alias("package_id_source"),
        F.col("recipient.customer_key").alias("recipient_customer_key"),
        F.col("p.recipient_customer_id").alias("recipient_customer_id_source"),
        F.col("sender_customer.customer_key").alias("sender_customer_key"),
        F.col("p.sender_customer_id").alias("sender_customer_id_source"),
        F.col("sender_business.business_key").alias("sender_business_key"),
        F.col("p.sender_business_id").alias("sender_business_id_source"),
        deterministic_key(normalized_str(F.col("p.service_type_id"))).alias("service_type_key"),
        F.col("p.service_type_id").alias("service_type_id_source"),
        F.col("st.service_type_name"),
        deterministic_key(normalized_str(F.col("p.package_status_id"))).alias("package_status_key"),
        F.col("p.package_status_id").alias("package_status_id_source"),
        F.col("ps.status_name").alias("package_status_name"),
        deterministic_key(normalized_str(F.col("p.package_flow_type_id"))).alias("package_flow_type_key"),
        F.col("p.package_flow_type_id").alias("package_flow_type_id_source"),
        F.col("pft.package_flow_type_name"),
        F.col("dp.planned_origin_facility_key").alias("origin_facility_key"),
        F.col("dp.planned_origin_facility_id_source").alias("origin_facility_id_source"),
        F.col("dp.planned_destination_facility_key").alias("destination_facility_key"),
        F.col("dp.planned_destination_facility_id_source").alias("destination_facility_id_source"),
        F.col("recipient_geo.geography_key").alias("recipient_geography_key"),
        F.col("sd.recipient_territory_id").alias("recipient_territory_id_source"),
        F.col("sender_geo.geography_key").alias("sender_geography_key"),
        F.col("sd.sender_territory_id").alias("sender_territory_id_source"),
        F.col("p.received_date").alias("received_at_origin_timestamp"),
        date_key_from_timestamp("p.received_date").alias("received_at_origin_date_key"),
        F.col("m.sorted_at_origin_timestamp"),
        date_key_from_timestamp("m.sorted_at_origin_timestamp").alias("sorted_at_origin_date_key"),
        F.col("m.departed_origin_timestamp"),
        date_key_from_timestamp("m.departed_origin_timestamp").alias("departed_origin_date_key"),
        F.col("sd.expected_delivery_date"),
        date_key_from_timestamp("sd.expected_delivery_date").alias("expected_delivery_date_key"),
        F.col("sd.delivered_date").alias("actual_delivery_timestamp"),
        date_key_from_timestamp("sd.delivered_date").alias("actual_delivery_date_key"),
        F.col("m.first_movement_timestamp"),
        F.col("m.last_movement_timestamp"),
        F.lit(1).alias("package_count"),
        F.coalesce(F.col("m.movement_count"), F.lit(0)).alias("movement_count"),
        F.coalesce(F.col("i.incident_count"), F.lit(0)).alias("incident_count"),
        F.coalesce(F.col("r.refund_count"), F.lit(0)).alias("refund_count"),
        F.when(status_name_norm == "delivered", F.lit(True)).otherwise(F.lit(False)).alias("is_delivered"),
        F.when((status_name_norm == "delayed") | (status_category_norm == "delayed"), F.lit(True)).otherwise(F.lit(False)).alias("is_delayed"),
        F.when((status_name_norm == "lost") | (status_name_norm == "lost package"), F.lit(True)).otherwise(F.lit(False)).alias("is_lost"),
        F.when((status_name_norm == "returned") | (status_name_norm == "returned package"), F.lit(True)).otherwise(F.lit(False)).alias("is_returned"),
        F.when((status_name_norm == "cancelled") | (status_name_norm == "canceled"), F.lit(True)).otherwise(F.lit(False)).alias("is_cancelled"),
        F.when(F.coalesce(F.col("i.incident_count"), F.lit(0)) > 0, F.lit(True)).otherwise(F.lit(False)).alias("has_incident"),
        F.when(F.coalesce(F.col("r.refund_count"), F.lit(0)) > 0, F.lit(True)).otherwise(F.lit(False)).alias("has_refund"),
        bool_from_tinyint(F.col("ps.is_final_status")).alias("is_final_status"), #Help function since MySQLL stores booleanns as TINYINT
        F.when(F.lower(F.coalesce(F.col("pft.package_flow_type_name"), F.lit(""))) == "b2c", F.lit(True)).otherwise(F.lit(False)).alias("is_b2c"),
        F.when(F.lower(F.coalesce(F.col("pft.package_flow_type_name"), F.lit(""))) == "p2p", F.lit(True)).otherwise(F.lit(False)).alias("is_p2p"),
        F.col("p.contents"),
        F.col("p.weight_oz"),
        F.col("p.length_in"),
        F.col("p.width_in"),
        F.col("p.height_in"),
        (F.col("p.length_in") * F.col("p.width_in") * F.col("p.height_in")).alias("package_volume_cubic_in"),
        F.when(
            F.col("sd.delivered_date").isNotNull() & F.col("p.received_date").isNotNull(),
            (F.col("sd.delivered_date").cast("long") - F.col("p.received_date").cast("long")) / F.lit(3600.0)
        ).otherwise(F.lit(None).cast("double")).alias("delivery_duration_hours"),
        F.when(
            F.col("m.departed_origin_timestamp").isNotNull() & F.col("p.received_date").isNotNull(),
            (F.col("m.departed_origin_timestamp").cast("long") - F.col("p.received_date").cast("long")) / F.lit(3600.0)
        ).otherwise(F.lit(None).cast("double")).alias("origin_processing_hours"),
        F.when(
            F.col("sd.delivered_date").isNotNull() & F.col("m.departed_origin_timestamp").isNotNull(),
            (F.col("sd.delivered_date").cast("long") - F.col("m.departed_origin_timestamp").cast("long")) / F.lit(3600.0)
        ).otherwise(F.lit(None).cast("double")).alias("transit_duration_hours"),
        F.col("sd.estimated_delivery_distance"),
        F.coalesce(F.col("sc.shipping_revenue_amount"), F.lit(0)).alias("shipping_revenue_amount"),
        F.coalesce(F.col("sc.material_cost_amount"), F.lit(0)).alias("material_cost_amount"),
        F.coalesce(F.col("sc.transportation_cost_amount"), F.lit(0)).alias("transportation_cost_amount"),
        (F.coalesce(F.col("sc.material_cost_amount"), F.lit(0)) + F.coalesce(F.col("sc.transportation_cost_amount"), F.lit(0))).alias("shipping_cost_amount"),
        F.coalesce(F.col("r.refund_amount_total"), F.lit(0)).alias("refund_amount"),
        (
            F.coalesce(F.col("sc.shipping_revenue_amount"), F.lit(0))
            - (F.coalesce(F.col("sc.material_cost_amount"), F.lit(0)) + F.coalesce(F.col("sc.transportation_cost_amount"), F.lit(0)))
        ).alias("gross_margin_amount"),
        (
            F.coalesce(F.col("sc.shipping_revenue_amount"), F.lit(0)) - F.coalesce(F.col("r.refund_amount_total"), F.lit(0))
        ).alias("net_revenue_amount"),
    )
)

fact_package = write_gold_table(fact_package, "fact_package")
add_table_row_count("fact_package", fact_package)

In [0]:
# Build fact_package_movement

movement_sequence_window = Window.partitionBy(F.col("pm.package_id")).orderBy(
    F.col("pm.event_timestamp").asc_nulls_last(),
    F.col("pm.package_movement_id")
)

fact_package_movement = with_gold_audit_columns(
    package_movement.alias("pm")
    .join(dim_package.alias("dp"), F.col("pm.package_id") == F.col("dp.package_id_source"), "left")
    .join(dim_facility.alias("event_facility"), F.col("pm.facility_id") == F.col("event_facility.facility_id_source"), "left")
    .join(dim_facility.alias("from_facility"), F.col("pm.from_facility_id") == F.col("from_facility.facility_id_source"), "left")
    .join(dim_facility.alias("to_facility"), F.col("pm.to_facility_id") == F.col("to_facility.facility_id_source"), "left")
    .join(dim_employee.alias("de"), F.col("pm.processed_by_employee_id") == F.col("de.employee_id_source"), "left")
    .join(
        dim_package_movement_event_type.alias("dmet"),
        F.col("pm.package_movement_event_type_id") == F.col("dmet.package_movement_event_type_id_source"),
        "left",
    )
    .join(package_status.alias("ps"), F.col("pm.package_status_id") == F.col("ps.package_status_id"), "left")
    .join(package.alias("p"), F.col("pm.package_id") == F.col("p.package_id"), "left")
    .join(service_type.alias("st"), F.col("p.service_type_id") == F.col("st.service_type_id"), "left")
    .join(package_flow_type.alias("pft"), F.col("p.package_flow_type_id") == F.col("pft.package_flow_type_id"), "left")
    .withColumn("movement_sequence", F.row_number().over(movement_sequence_window))
    .select(
        deterministic_key(normalized_str(F.col("pm.package_movement_id"))).alias("fact_package_movement_key"),
        F.col("pm.package_movement_id").alias("package_movement_id_source"),
        F.col("dp.package_key"),
        F.col("pm.package_id").alias("package_id_source"),
        F.col("event_facility.facility_key").alias("facility_key"),
        F.col("pm.facility_id").alias("facility_id_source"),
        F.col("from_facility.facility_key").alias("from_facility_key"),
        F.col("pm.from_facility_id").alias("from_facility_id_source"),
        F.col("to_facility.facility_key").alias("to_facility_key"),
        F.col("pm.to_facility_id").alias("to_facility_id_source"),
        F.col("de.employee_key").alias("employee_key"),
        F.col("pm.processed_by_employee_id").alias("employee_id_source"),
        F.col("de.department_key").alias("department_key"),
        F.col("de.department_id_source").alias("department_id_source"),
        deterministic_key(normalized_str(F.col("p.service_type_id"))).alias("service_type_key"),
        F.col("p.service_type_id").alias("service_type_id_source"),
        F.col("st.service_type_name"),
        deterministic_key(normalized_str(F.col("p.package_flow_type_id"))).alias("package_flow_type_key"),
        F.col("p.package_flow_type_id").alias("package_flow_type_id_source"),
        deterministic_key(normalized_str(F.col("pm.package_status_id"))).alias("package_status_key"),
        F.col("pm.package_status_id").alias("package_status_id_source"),
        F.col("pm.package_movement_event_type_id").alias("package_movement_event_type_id_source"),
        F.col("dmet.package_movement_event_type_key"),
        F.col("pm.event_timestamp"),
        date_key_from_timestamp("pm.event_timestamp").alias("movement_date_key"),
        F.col("movement_sequence"),
        F.col("pm.movement_note"),
        F.when(F.lower(F.coalesce(F.col("pft.package_flow_type_name"), F.lit(""))) == "b2c", F.lit(True)).otherwise(F.lit(False)).alias("is_b2c"),
        F.when(F.lower(F.coalesce(F.col("pft.package_flow_type_name"), F.lit(""))) == "p2p", F.lit(True)).otherwise(F.lit(False)).alias("is_p2p"),
        F.lit(1).alias("movement_count"),
    )
)

fact_package_movement = write_gold_table(fact_package_movement, "fact_package_movement")
add_table_row_count("fact_package_movement", fact_package_movement)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4525154949166083>, line 62
      3 movement_sequence_window = Window.partitionBy(F.col("pm.package_id")).orderBy(
      4     F.col("pm.event_timestamp").asc_nulls_last(),
      5     F.col("pm.package_movement_id")
      6 )
      8 fact_package_movement = with_gold_audit_columns(
      9     package_movement.alias("pm")
     10     .join(dim_package.alias("dp"), F.col("pm.package_id") == F.col("dp.package_id_source"), "left")
   (...)
     59     )
     60 )
---> 62 fact_package_movement = write_gold_table(fact_package_movement, "fact_package_movement")
     63 add_table_row_count("fact_package_movement", fact_package_movement)

File <command-4525154949166073>, line 102, in write_gold_table(df, table_name)
     94 def write_gold_table(df: DataFrame, table_name: str) -> DataFrame:
     95     table_path = f"{gold_base_pat

In [0]:
# Build fact_shipping_revenue_cost

refund_agg_for_shipping = refunds.groupBy("package_id").agg(F.sum("refund_amount").alias("refund_amount_total"))

fact_shipping_revenue_cost = with_gold_audit_columns(
    shipping_cost.alias("sc")
    .join(dim_package.alias("dp"), F.col("sc.package_id") == F.col("dp.package_id_source"), "left")
    .join(package.alias("p"), F.col("sc.package_id") == F.col("p.package_id"), "left")
    .join(shippingdetails.alias("sd"), F.col("sc.package_id") == F.col("sd.package_id"), "left")
    .join(refund_agg_for_shipping.alias("r"), F.col("sc.package_id") == F.col("r.package_id"), "left")
    .join(dim_customer.alias("recipient"), F.col("p.recipient_customer_id") == F.col("recipient.customer_id_source"), "left")
    .join(dim_customer.alias("sender_customer"), F.col("p.sender_customer_id") == F.col("sender_customer.customer_id_source"), "left")
    .join(dim_business.alias("sender_business"), F.col("p.sender_business_id") == F.col("sender_business.business_id_source"), "left")
    .join(dim_geography.alias("dest_geo"), F.col("sd.recipient_territory_id") == F.col("dest_geo.territory_id_source"), "left")
    .join(dim_geography.alias("sender_geo"), F.col("sd.sender_territory_id") == F.col("sender_geo.territory_id_source"), "left")
    .select(
        deterministic_key(
            normalized_binary(F.col("sc.package_id")),
            normalized_str(F.col("sc.charge_recorded_at")),
            normalized_str(F.col("sc.charge_source")),
        ).alias("fact_shipping_revenue_cost_key"),
        F.col("dp.package_key"),
        F.col("sc.package_id").alias("package_id_source"),
        F.col("recipient.customer_key").alias("recipient_customer_key"),
        F.col("p.recipient_customer_id").alias("recipient_customer_id_source"),
        F.col("sender_customer.customer_key").alias("sender_customer_key"),
        F.col("p.sender_customer_id").alias("sender_customer_id_source"),
        F.col("sender_business.business_key").alias("sender_business_key"),
        F.col("p.sender_business_id").alias("sender_business_id_source"),
        deterministic_key(normalized_str(F.col("p.service_type_id"))).alias("service_type_key"),
        F.col("p.service_type_id").alias("service_type_id_source"),
        deterministic_key(normalized_str(F.col("p.package_flow_type_id"))).alias("package_flow_type_key"),
        F.col("p.package_flow_type_id").alias("package_flow_type_id_source"),
        F.col("dp.planned_origin_facility_key").alias("origin_facility_key"),
        F.col("dp.planned_origin_facility_id_source").alias("origin_facility_id_source"),
        F.col("dp.planned_destination_facility_key").alias("destination_facility_key"),
        F.col("dp.planned_destination_facility_id_source").alias("destination_facility_id_source"),
        F.col("dest_geo.geography_key").alias("destination_geography_key"),
        F.col("sd.recipient_territory_id").alias("destination_territory_id_source"),
        F.col("sender_geo.geography_key").alias("sender_geography_key"),
        F.col("sd.sender_territory_id").alias("sender_territory_id_source"),
        F.col("sc.charge_recorded_at").alias("calculation_timestamp"),
        date_key_from_timestamp("sc.charge_recorded_at").alias("calculation_date_key"),
        F.col("sd.estimated_delivery_distance").alias("distance_miles"),
        F.col("p.weight_oz"),
        (F.col("p.length_in") * F.col("p.width_in") * F.col("p.height_in")).alias("package_volume_cubic_in"),
        F.col("sc.actual_shipping_charge").alias("total_revenue_amount"),
        F.col("sc.material_cost").alias("material_cost_amount"),
        F.col("sc.transportation_cost").alias("transportation_cost_amount"),
        (F.coalesce(F.col("sc.material_cost"), F.lit(0)) + F.coalesce(F.col("sc.transportation_cost"), F.lit(0))).alias("total_cost_amount"),
        (
            F.coalesce(F.col("sc.actual_shipping_charge"), F.lit(0))
            - (F.coalesce(F.col("sc.material_cost"), F.lit(0)) + F.coalesce(F.col("sc.transportation_cost"), F.lit(0)))
        ).alias("gross_margin_amount"),
        F.coalesce(F.col("r.refund_amount_total"), F.lit(0)).alias("refund_amount"),
        (F.coalesce(F.col("sc.actual_shipping_charge"), F.lit(0)) - F.coalesce(F.col("r.refund_amount_total"), F.lit(0))).alias("net_revenue_amount"),
        F.col("sc.charge_source"),
        F.lit(1).alias("shipping_cost_record_count"),
    )
)

fact_shipping_revenue_cost = write_gold_table(fact_shipping_revenue_cost, "fact_shipping_revenue_cost")
add_table_row_count("fact_shipping_revenue_cost", fact_shipping_revenue_cost)

In [0]:
# Build fact_incident

fact_incident = with_gold_audit_columns(
    incident.alias("i")
    .join(dim_package.alias("dp"), F.col("i.package_id") == F.col("dp.package_id_source"), "left")
    .join(dim_incident_type.alias("dit"), F.col("i.incident_type_id") == F.col("dit.incident_type_id_source"), "left")
    .join(dim_employee.alias("reported_emp"), F.col("i.reported_by_employee_id") == F.col("reported_emp.employee_id_source"), "left")
    .join(dim_employee.alias("resolved_emp"), F.col("i.resolved_by_employee_id") == F.col("resolved_emp.employee_id_source"), "left")
    .join(dim_facility.alias("df"), F.col("i.facility_id") == F.col("df.facility_id_source"), "left")
    .join(
        dim_incident_status.alias("dis"),
        F.col("i.incident_status_id") == F.col("dis.incident_status_id_source"),
        "left"
    )
    .join(
        dim_incident_severity.alias("dsev"),
        F.col("i.incident_severity_id") == F.col("dsev.incident_severity_id_source"),
        "left"
    )
    .join(dim_customer.alias("dc"), F.col("i.customer_id") == F.col("dc.customer_id_source"), "left")
    .select(
        deterministic_key(normalized_str(F.col("i.incident_id"))).alias("fact_incident_key"),
        F.col("i.incident_id").alias("incident_id_source"),
        F.col("dp.package_key"),
        F.col("i.package_id").alias("package_id_source"),
        F.col("dit.incident_type_key"),
        F.col("i.incident_type_id").alias("incident_type_id_source"),
        F.col("reported_emp.employee_key").alias("reported_by_employee_key"),
        F.col("i.reported_by_employee_id").alias("reported_by_employee_id_source"),
        F.col("resolved_emp.employee_key").alias("resolved_by_employee_key"),
        F.col("i.resolved_by_employee_id").alias("resolved_by_employee_id_source"),
        F.col("reported_emp.department_key").alias("reported_department_key"),
        F.col("reported_emp.department_id_source").alias("reported_department_id_source"),
        F.col("df.facility_key"),
        F.col("i.facility_id").alias("facility_id_source"),
        F.col("dc.customer_key").alias("customer_key"),
        F.col("i.customer_id").alias("customer_id_source"),
        F.col("i.package_movement_id").alias("package_movement_id_source"),
        F.col("i.incident_date"),
        date_key_from_timestamp("i.incident_date").alias("incident_date_key"),
        F.col("i.resolved_at"),
        date_key_from_timestamp("i.resolved_at").alias("resolved_date_key"),
        F.col("i.description").alias("incident_description"),
        F.col("i.resolution_note"),
        ((F.col("i.resolved_at").cast("long") - F.col("i.incident_date").cast("long")) / F.lit(3600.0)).alias("resolution_duration_hours"),
        F.when(F.col("i.resolved_at").isNotNull(), F.lit(True)).otherwise(F.lit(False)).alias("is_resolved"),
        F.col("dis.incident_status_key"),
        F.col("i.incident_status_id").alias("incident_status_id_source"),
        F.col("dsev.incident_severity_key"),
        F.col("i.incident_severity_id").alias("incident_severity_id_source"),
        F.lit(1).alias("incident_count"),
    )
)

fact_incident = write_gold_table(fact_incident, "fact_incident")
add_table_row_count("fact_incident", fact_incident)

In [0]:
# Build fact_refund

fact_refund = with_gold_audit_columns(
    refunds.alias("r")
    .join(incident.alias("i"), F.col("r.incident_id") == F.col("i.incident_id"), "left")
    .join(dim_package.alias("dp"), F.col("r.package_id") == F.col("dp.package_id_source"), "left")
    .join(dim_refund_status.alias("drs"), F.trim(F.col("r.refund_status")) == F.col("drs.refund_status_name"), "left")
    .join(dim_incident_type.alias("dit"), F.col("i.incident_type_id") == F.col("dit.incident_type_id_source"), "left")
    .join(dim_customer.alias("dc"), F.col("r.customer_id") == F.col("dc.customer_id_source"), "left")
    .join(dim_employee.alias("reviewer"), F.col("r.reviewed_by_employee_id") == F.col("reviewer.employee_id_source"), "left")
    .join(dim_facility.alias("df"), F.col("i.facility_id") == F.col("df.facility_id_source"), "left")
    .select(
        deterministic_key(normalized_str(F.col("r.refund_id"))).alias("fact_refund_key"),
        F.col("r.refund_id").alias("refund_id_source"),
        F.col("dp.package_key"),
        F.col("r.package_id").alias("package_id_source"),
        F.col("r.incident_id").alias("incident_id_source"),
        F.col("i.package_movement_id").alias("package_movement_id_source"),
        F.col("drs.refund_status_key"),
        F.trim(F.col("r.refund_status")).alias("refund_status_id_source"),
        F.col("drs.refund_status_name"),
        F.col("dit.incident_type_key"),
        F.col("i.incident_type_id").alias("incident_type_id_source"),
        F.col("dit.incident_type_name"),
        F.col("dc.customer_key").alias("customer_key"),
        F.col("r.customer_id").alias("customer_id_source"),
        F.col("reviewer.employee_key").alias("reviewed_by_employee_key"),
        F.col("r.reviewed_by_employee_id").alias("reviewed_by_employee_id_source"),
        F.col("reviewer.department_key").alias("reviewed_by_department_key"),
        F.col("df.facility_key"),
        F.col("i.facility_id").alias("facility_id_source"),
        F.col("r.refund_date").alias("requested_timestamp"),
        date_key_from_timestamp("r.refund_date").alias("requested_date_key"),
        F.col("r.reviewed_at").alias("decision_timestamp"),
        date_key_from_timestamp("r.reviewed_at").alias("decision_date_key"),
        F.col("r.paid_at").alias("disbursed_timestamp"),
        date_key_from_timestamp("r.paid_at").alias("disbursed_date_key"),
        F.col("r.refund_amount").alias("requested_amount"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))).isin(["approved", "paid"]), F.col("r.refund_amount")).otherwise(F.lit(0)).alias("approved_amount"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))) == "paid", F.col("r.refund_amount")).otherwise(F.lit(0)).alias("disbursed_amount"),
        ((F.col("r.reviewed_at").cast("long") - F.col("r.refund_date").cast("long")) / F.lit(3600.0)).alias("decision_duration_hours"),
        ((F.col("r.paid_at").cast("long") - F.col("r.refund_date").cast("long")) / F.lit(3600.0)).alias("disbursement_duration_hours"),
        F.col("r.refund_reason"),
        F.col("r.refund_note"),
        F.when(F.col("r.refund_date").isNotNull(), F.lit(True)).otherwise(F.lit(False)).alias("is_requested"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))) == "approved", F.lit(True)).otherwise(F.lit(False)).alias("is_approved"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))) == "rejected", F.lit(True)).otherwise(F.lit(False)).alias("is_rejected"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))) == "paid", F.lit(True)).otherwise(F.lit(False)).alias("is_disbursed"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))) == "pending", F.lit(True)).otherwise(F.lit(False)).alias("is_pending"),
        F.when(F.lower(F.coalesce(F.col("r.refund_status"), F.lit(""))).isin(["approved", "rejected", "paid", "cancelled"]), F.lit(True)).otherwise(F.lit(False)).alias("is_final_status"),
        F.lit(1).alias("refund_count"),
    )
)

fact_refund = write_gold_table(fact_refund, "fact_refund")
add_table_row_count("fact_refund", fact_refund)

In [0]:
# Build fact_smartlocker_assignment

smartlocker_assignment_base = (
    package_to_locker.alias("ptl")
    .join(lockerassignment.alias("la"), F.col("ptl.locker_assignment_id") == F.col("la.locker_assignment_id"), "left")
    .join(smartlocker.alias("sl"), F.col("la.locker_id") == F.col("sl.locker_id"), "left")
)

if lockerlocations is not None:
    smartlocker_assignment_base = (
        smartlocker_assignment_base.join(
            lockerlocations.alias("ll"),
            F.col("sl.locker_location_id") == F.col("ll.locker_location_id"),
            "left",
        )
        .join(dim_facility.alias("df"), F.col("ll.facility_id") == F.col("df.facility_id_source"), "left")
        # Resolve facility/geography columns in the optional lockerlocations branch.
        .withColumn("resolved_facility_key", F.col("df.facility_key"))
        .withColumn("resolved_facility_id_source", F.col("df.facility_id_source"))
        .withColumn("resolved_geography_key", F.col("df.geography_key"))
    )
else:
    smartlocker_assignment_base = (
        smartlocker_assignment_base.withColumn("resolved_facility_key", F.lit(None).cast("long"))
        .withColumn("resolved_facility_id_source", F.lit(None).cast("int"))
        .withColumn("resolved_geography_key", F.lit(None).cast("long"))
    )

fact_smartlocker_assignment = with_gold_audit_columns(
    smartlocker_assignment_base
    .join(dim_package.alias("dp"), F.col("ptl.package_id") == F.col("dp.package_id_source"), "left")
    .join(dim_customer.alias("dc"), F.coalesce(F.col("ptl.customer_id"), F.col("la.customer_id")) == F.col("dc.customer_id_source"), "left")
    .join(dim_smartlocker.alias("dsl"), F.col("sl.locker_id") == F.col("dsl.locker_id_source"), "left")
    .withColumn(
        "assignment_status_name",
        F.when(F.col("la.retrieved_at").isNotNull(), F.lit("picked_up"))
        .when(F.col("la.expires_at").isNotNull(), F.lit("awaiting_pickup"))
        .otherwise(F.lit("assigned"))
    )
    .join(
        dim_smartlocker_assignment_status.alias("dsas"),
        F.col("assignment_status_name") == F.col("dsas.assignment_status_name"),
        "left"
    )
    .select(
        deterministic_key(
            normalized_binary(F.col("ptl.package_id")),
            normalized_str(F.col("ptl.locker_assignment_id"))
        ).alias("fact_smartlocker_assignment_key"),
        F.col("dp.package_key"),
        F.col("ptl.package_id").alias("package_id_source"),
        F.col("dsl.smartlocker_key").alias("smartlocker_key"),
        F.col("sl.locker_id").alias("locker_id_source"),
        F.col("ptl.locker_assignment_id").alias("locker_assignment_id_source"),
        F.col("dc.customer_key").alias("customer_key"),
        F.coalesce(F.col("ptl.customer_id"), F.col("la.customer_id")).alias("customer_id_source"),
        F.col("resolved_facility_key").alias("facility_key"),
        F.col("resolved_facility_id_source").alias("facility_id_source"),
        F.col("resolved_geography_key").alias("geography_key"),
        F.col("ptl.assigned_at").alias("assigned_timestamp"),
        date_key_from_timestamp("ptl.assigned_at").alias("assigned_date_key"),
        # TODO: No separate placed_at column exists in Silver, so assigned_at is the closest available placement timestamp.
        F.col("ptl.assigned_at").alias("placed_timestamp"),
        F.col("la.expires_at").alias("pickup_due_timestamp"),
        date_key_from_timestamp("la.expires_at").alias("pickup_due_date_key"),
        F.col("la.retrieved_at").alias("picked_up_timestamp"),
        date_key_from_timestamp("la.retrieved_at").alias("picked_up_date_key"),
        ((F.col("la.retrieved_at").cast("long") - F.col("ptl.assigned_at").cast("long")) / F.lit(3600.0)).alias("time_to_pickup_hours"),
        ((F.col("la.expires_at").cast("long") - F.col("ptl.assigned_at").cast("long")) / F.lit(3600.0)).alias("locker_dwell_hours_until_due"),
        F.col("dsas.assignment_status_key"),
        F.when(F.col("la.expires_at").isNotNull() & F.col("la.retrieved_at").isNull() & (F.col("la.expires_at") < F.current_timestamp()), F.lit(True)).otherwise(F.lit(False)).alias("is_overdue_pickup"),
        F.lit(1).alias("assignment_count"),
    )
)

fact_smartlocker_assignment = write_gold_table(fact_smartlocker_assignment, "fact_smartlocker_assignment")
add_table_row_count("fact_smartlocker_assignment", fact_smartlocker_assignment)

In [0]:
# Fact validation

expected_source_counts = {
    "fact_package": source_table_counts["package"],
    "fact_package_movement": source_table_counts["package_movement"],
    "fact_shipping_revenue_cost": source_table_counts["shipping_cost"],
    "fact_incident": source_table_counts["incident"],
    "fact_refund": source_table_counts["refunds"],
    "fact_smartlocker_assignment": source_table_counts["package_to_locker"],
}

fact_frames = {
    "fact_package": fact_package,
    "fact_package_movement": fact_package_movement,
    "fact_shipping_revenue_cost": fact_shipping_revenue_cost,
    "fact_incident": fact_incident,
    "fact_refund": fact_refund,
    "fact_smartlocker_assignment": fact_smartlocker_assignment,
}

for fact_name, fact_df in fact_frames.items():
    actual_count = fact_df.count()
    validate_row_count(fact_name, actual_count, expected_source_counts[fact_name])

validate_unique_keys(fact_package, "fact_package", ["fact_package_key"])
validate_unique_keys(fact_package, "fact_package", ["package_id_source"], check_name="duplicate_source_grain_check")

validate_unique_keys(fact_package_movement, "fact_package_movement", ["fact_package_movement_key"])
validate_unique_keys(fact_package_movement, "fact_package_movement", ["package_movement_id_source"], check_name="duplicate_source_grain_check")

validate_unique_keys(fact_shipping_revenue_cost, "fact_shipping_revenue_cost", ["fact_shipping_revenue_cost_key"])

validate_unique_keys(fact_incident, "fact_incident", ["fact_incident_key"])
validate_unique_keys(fact_incident, "fact_incident", ["incident_id_source"], check_name="duplicate_source_grain_check")

validate_unique_keys(fact_refund, "fact_refund", ["fact_refund_key"])
validate_unique_keys(fact_refund, "fact_refund", ["refund_id_source"], check_name="duplicate_source_grain_check")

validate_unique_keys(fact_smartlocker_assignment, "fact_smartlocker_assignment", ["fact_smartlocker_assignment_key"])
validate_unique_keys(
    fact_smartlocker_assignment,
    "fact_smartlocker_assignment",
    ["package_id_source", "locker_assignment_id_source"],
    check_name="duplicate_source_grain_check",
)

for fact_name, required_keys in {
    "fact_package": ["package_key"],

    "fact_package_movement": [
        "package_key",
        "package_movement_event_type_key"
    ],

    "fact_shipping_revenue_cost": ["package_key"],

    "fact_incident": [
        "package_key",
        "incident_type_key",
        "incident_status_key",
        "incident_severity_key"
    ],

    "fact_refund": [
        "package_key",
        "refund_status_key"
    ],

    "fact_smartlocker_assignment": [
        "package_key",
        "smartlocker_key",
        "assignment_status_key"
    ],
}.items():
    for required_key in required_keys:
        validate_required_foreign_key(fact_frames[fact_name], fact_name, required_key)

record_validation(
    check_name="known_edge_case",
    table_name="fact_shipping_revenue_cost",
    status="PASS" if source_table_counts["package"] - source_table_counts["shipping_cost"] == 12 else "WARN",
    metric_value=source_table_counts["package"] - source_table_counts["shipping_cost"],
    details="shipping_cost source has 12 fewer rows than package source",
)

refunds_missing_incident_id = fact_refund.filter(F.col("incident_id_source").isNull()).count()
record_validation(
    check_name="refund_missing_incident_id_check",
    table_name="fact_refund",
    status="PASS" if refunds_missing_incident_id == 0 else "WARN",
    metric_value=refunds_missing_incident_id,
    details="Refund records should normally reference a valid incident_id in the postal business model.",
)

refunds_unresolved_incident = fact_refund.filter(
    F.col("incident_id_source").isNotNull() & F.col("incident_type_key").isNull()
).count()
record_validation(
    check_name="refund_unresolved_incident_check",
    table_name="fact_refund",
    status="PASS" if refunds_unresolved_incident == 0 else "WARN",
    metric_value=refunds_unresolved_incident,
    details="Refunds with an incident_id should resolve incident attributes where source data supports the relationship.",
)

display(spark.createDataFrame(validation_results).orderBy("table_name", "check_name"))

In [0]:
# Relationship / orphan checks

validate_orphan_keys(fact_package, "fact_package", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_package, "fact_package", "recipient_customer_key", dim_customer, "customer_key", "dim_customer")
validate_orphan_keys(fact_package, "fact_package", "sender_customer_key", dim_customer, "customer_key", "dim_customer")
validate_orphan_keys(fact_package, "fact_package", "sender_business_key", dim_business, "business_key", "dim_business")
validate_orphan_keys(fact_package, "fact_package", "origin_facility_key", dim_facility, "facility_key", "dim_facility")
validate_orphan_keys(fact_package, "fact_package", "destination_facility_key", dim_facility, "facility_key", "dim_facility")
validate_orphan_keys(fact_package, "fact_package", "recipient_geography_key", dim_geography, "geography_key", "dim_geography")
validate_orphan_keys(fact_package_movement, "fact_package_movement", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_package_movement, "fact_package_movement", "facility_key", dim_facility, "facility_key", "dim_facility")
validate_orphan_keys(fact_package_movement, "fact_package_movement", "from_facility_key", dim_facility, "facility_key", "dim_facility")
validate_orphan_keys(fact_package_movement, "fact_package_movement", "to_facility_key", dim_facility, "facility_key", "dim_facility")
validate_orphan_keys(fact_package_movement, "fact_package_movement", "employee_key", dim_employee, "employee_key", "dim_employee")
validate_orphan_keys(fact_shipping_revenue_cost, "fact_shipping_revenue_cost", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_shipping_revenue_cost, "fact_shipping_revenue_cost", "destination_geography_key", dim_geography, "geography_key", "dim_geography")
validate_orphan_keys(fact_incident, "fact_incident", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_incident, "fact_incident", "incident_type_key", dim_incident_type, "incident_type_key", "dim_incident_type")
validate_orphan_keys(fact_refund, "fact_refund", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_refund, "fact_refund", "refund_status_key", dim_refund_status, "refund_status_key", "dim_refund_status")
validate_orphan_keys(fact_refund, "fact_refund", "incident_type_key", dim_incident_type, "incident_type_key", "dim_incident_type")
validate_orphan_keys(fact_smartlocker_assignment, "fact_smartlocker_assignment", "package_key", dim_package, "package_key", "dim_package")
validate_orphan_keys(fact_smartlocker_assignment, "fact_smartlocker_assignment", "smartlocker_key", dim_smartlocker, "smartlocker_key", "dim_smartlocker")

validate_orphan_keys(
    fact_package_movement,
    "fact_package_movement",
    "package_movement_event_type_key",
    dim_package_movement_event_type,
    "package_movement_event_type_key",
    "dim_package_movement_event_type"
)

validate_orphan_keys(
    fact_incident,
    "fact_incident",
    "incident_status_key",
    dim_incident_status,
    "incident_status_key",
    "dim_incident_status"
)

validate_orphan_keys(
    fact_incident,
    "fact_incident",
    "incident_severity_key",
    dim_incident_severity,
    "incident_severity_key",
    "dim_incident_severity"
)

validate_orphan_keys(
    fact_smartlocker_assignment,
    "fact_smartlocker_assignment",
    "assignment_status_key",
    dim_smartlocker_assignment_status,
    "assignment_status_key",
    "dim_smartlocker_assignment_status"
)

display(spark.createDataFrame(orphan_results).orderBy("fact_table", "fact_column"))

In [0]:
# Optional optimize / maintenance

if enable_optimize:
    for table_name in gold_tables_created:
        spark.sql(f"OPTIMIZE {fqtn(table_name)}")

if enable_vacuum:
    # TODO: VACUUM remains disabled by default because it is operationally unsafe for an initial rerunnable build notebook.
    for table_name in gold_tables_created:
        spark.sql(f"VACUUM {fqtn(table_name)} RETAIN 168 HOURS")

In [0]:
# Final Gold summary

gold_table_summary = spark.createDataFrame(table_row_counts).orderBy("table_name")
validation_summary_df = spark.createDataFrame(validation_results).orderBy("table_name", "check_name")
orphan_summary_df = spark.createDataFrame(orphan_results).orderBy("fact_table", "fact_column")
warnings_df = spark.createDataFrame(warnings_log) if warnings_log else spark.createDataFrame([], "warning_message string")

print("Gold tables created:")
for table_name in gold_tables_created:
    print(f" - {catalog_name}.{gold_schema}.{table_name}")

print("")
print("Power BI next-step recommendations:")
print(" - Use one-to-many relationships from dimensions to facts on surrogate keys.")
print(" - Use dim_date as the primary date table and role-play it in Power BI for event-specific date keys.")
print(" - Keep fact_package as the lifecycle summary fact and fact_package_movement as the detailed operational event fact.")
print(" - Treat recipient/sender customer roles as separate semantic relationships in the BI model.")
print(" - Review warnings and validation outputs before promoting the Gold layer.")

display(gold_table_summary.limit(display_row_limit))
display(validation_summary_df.limit(display_row_limit))
display(orphan_summary_df.limit(display_row_limit))
display(warnings_df.limit(display_row_limit))

In [0]:
# Verify Gold table storage locations

from pyspark.sql.types import StructType, StructField, StringType, LongType

verification_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("format", StringType(), True),
    StructField("location", StringType(), True),
    StructField("num_files", LongType(), True),
    StructField("size_in_bytes", LongType(), True),
    StructField("error", StringType(), True),
])

rows = []

tables = spark.sql(f"SHOW TABLES IN `{catalog_name}`.`{gold_schema}`").collect()

for table in tables:
    table_name = table.tableName
    full_table_name = f"`{catalog_name}`.`{gold_schema}`.`{table_name}`"

    try:
        detail = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0].asDict()

        rows.append({
            "table_name": table_name,
            "format": str(detail.get("format")) if detail.get("format") is not None else None,
            "location": str(detail.get("location")) if detail.get("location") is not None else None,
            "num_files": int(detail.get("numFiles")) if detail.get("numFiles") is not None else None,
            "size_in_bytes": int(detail.get("sizeInBytes")) if detail.get("sizeInBytes") is not None else None,
            "error": None
        })

    except Exception as e:
        rows.append({
            "table_name": table_name,
            "format": None,
            "location": None,
            "num_files": None,
            "size_in_bytes": None,
            "error": str(e)
        })

gold_locations_df = spark.createDataFrame(rows, schema=verification_schema)

display(gold_locations_df.orderBy("table_name"))